In [23]:
import os
import gc
import time
import math
import json
import warnings
import threading
import itertools
import re
import sqlite3
import hashlib
import queue

from functools import lru_cache
from collections import deque
from shapely.strtree import STRtree
from pathlib import Path
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
    wait,
    FIRST_COMPLETED,
)

import numpy as np
import pandas as pd
import requests

import rasterio
from rasterio.enums import MaskFlags, Resampling
from rasterio.windows import Window

import shapely
import shapely.geometry

import pystac_client
import pystac

from pyproj import Transformer
from scipy.spatial import cKDTree

import zarr
from zarr.codecs import BloscCodec, BloscShuffle

from tqdm.auto import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pystac_client.stac_api_io import StacApiIO


In [24]:
# ============================================================
# CONFIGURATION — LOW-MEMORY AZURE PROFILE
# ============================================================

PC_API_URL = "https://planetarycomputer.microsoft.com/api"
PC_STAC_URL = f"{PC_API_URL}/stac/v1"


def _normalize_pc_stac_url(value):
    """Accept a plain URL or Markdown-linked URL, but return the canonical PC endpoint."""
    text = str(value).strip()
    markdown = re.fullmatch(r"\[([^\]]+)\]\((https?://[^)]+)\)", text)
    if markdown:
        text = markdown.group(2).strip()
    text = text.strip("<> ").rstrip("/")
    expected = "https://planetarycomputer.microsoft.com/api/stac/v1"
    if text != expected:
        raise ValueError(
            "Unexpected Planetary Computer STAC URL. Expected the canonical HTTPS "
            f"endpoint {expected!r}; got {text!r}."
        )
    return expected


PC_STAC_URL = _normalize_pc_stac_url(PC_STAC_URL)
PC_SAS_URL = f"{PC_API_URL}/sas/v1/token"

S1_COLLECTION = "sentinel-1-rtc"

# RESUME MODE: use the existing, audited inventory and overview-shard namespace.
# This notebook is intentionally tied to the established v1 paths; do not point it
# at a fresh namespace unless you intend to rebuild inventory and all scene shards.
OUTPUT_DIR = Path(
    "Data/Global/pipeline_stage1/sentinel1"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Production aggregation method selected from the benchmark.
# "overview" is much faster but is approximate relative to exact native
# 10 m -> 1 km means. Set to "exact" to preserve the native-resolution mean.
PRODUCTION_READ_STRATEGY = "overview"  # allowed: "overview", "exact"
if PRODUCTION_READ_STRATEGY not in {"overview", "exact"}:
    raise ValueError("PRODUCTION_READ_STRATEGY must be 'overview' or 'exact'.")

# Reuse the established cache paths. Resume mode validates the finished SQLite
# inventory and does not re-query completed clusters.
LEGACY_STAC_CACHE_DIR = OUTPUT_DIR / "stac_cache_v4"
STAC_CACHE_DIR = OUTPUT_DIR / "stac_cache_nearest_v1"
STAC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# SQLite replaces the repeatedly rewritten global JSON scene index.
SCENE_INDEX_DB_PATH = OUTPUT_DIR / "scene_inventory_nearest_v1.sqlite"

# Keep output namespaces strategy-specific: do not mix exact values with
# overview-resampled values, even though their array shapes are identical.
if PRODUCTION_READ_STRATEGY == "overview":
    SCENE_OBS_DIR = OUTPUT_DIR / "scene_observations_nearest_overview_v1"
    FINAL_OUTPUT_DIR = OUTPUT_DIR / "nearest_valid_overview_v1"
    SCHEMA_VERSION = 6
else:
    # Preserve the old exact-read path, including its completed scene shards.
    SCENE_OBS_DIR = OUTPUT_DIR / "scene_observations_nearest_v1"
    FINAL_OUTPUT_DIR = OUTPUT_DIR / "nearest_valid_v5"
    SCHEMA_VERSION = 5

SCENE_OBS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# A small persistent manifest records shards safely moved off the VM. The NPZ files
# themselves stay on the PC archive; restore only the shards needed for finalization.
SCENE_TRANSFER_MANIFEST_PATH = OUTPUT_DIR / "scene_transfer_manifest_nearest_v1.json"
RESTORE_LIST_DIR = OUTPUT_DIR / "restore_lists"
RESTORE_LIST_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Spatial grid
# ------------------------------------------------------------
PATCH_SIZE = 13
MODEL_RESOLUTION_M = 1000
S1_RESOLUTION_M = 10
PATCH_WIDTH_M = PATCH_SIZE * MODEL_RESOLUTION_M
PATCH_RADIUS_M = PATCH_WIDTH_M / 2
SOURCE_PATCH_PX = int(round(PATCH_WIDTH_M / S1_RESOLUTION_M))
HALF_PATCH_PX = SOURCE_PATCH_PX // 2
SOURCE_PIXELS_PER_MODEL_CELL = SOURCE_PATCH_PX // PATCH_SIZE
assert SOURCE_PATCH_PX % PATCH_SIZE == 0

# ------------------------------------------------------------
# Nearest-acquisition temporal rule
# ------------------------------------------------------------
# Select the closest acquisition with valid data, before OR after the sample
# date. The limit is a search tolerance, not a rolling/median window.
MAX_NEAREST_DAYS = 18
# Retained for the storage-aware clustering score, which uses temporal overlap.
S1_WINDOW_DAYS = MAX_NEAREST_DAYS
# A scene/polarization pair is considered usable when at least this many of
# the 169 1-km cells have a jointly valid value across the pair.
MIN_VALID_PAIR_PIXELS = 1
SOURCE_OFFSET_MISSING = int(np.iinfo(np.int16).max)

# ------------------------------------------------------------
# Polarizations
# ------------------------------------------------------------
# Keep the original four-band shard/output layout so existing v9 checkpoints
# remain readable, but only download VV and VH going forward.
REAL_BANDS = ["vv", "vh", "hh", "hv"]
DERIVED_BANDS = ["vv_vh_ratio"]
OUTPUT_BANDS = REAL_BANDS + DERIVED_BANDS
N_REAL_BANDS = len(REAL_BANDS)
N_OUTPUT_BANDS = len(OUTPUT_BANDS)
POLARIZATIONS = tuple(REAL_BANDS)  # stored-band schema, retained for compatibility
REQUIRED_POLARIZATIONS = ("vv", "vh")
READ_POLARIZATIONS = REQUIRED_POLARIZATIONS

# ------------------------------------------------------------
# Storage-aware clustering
# ------------------------------------------------------------
MAX_CLUSTER_SIDE_KM = 50.0
MAX_CLUSTER_STATIONS = 100
COG_BLOCK_PX = 512
COG_BLOCK_M = COG_BLOCK_PX * S1_RESOLUTION_M / 1000.0
MERGE_SLACK = 1.10

# ------------------------------------------------------------
# RAM-aware concurrency and retries
# ------------------------------------------------------------
# Estimate a conservative worker ceiling from VM RAM, not from the old 2-vCPU /
# 4-GB profile. Each scene worker can hold several raster patches and GDAL buffers.
# Reserve 3 GiB for the OS/kernel/notebook; budget ~2.5 GiB per scene worker.
def _detect_total_ram_gib():
    try:
        with open("/proc/meminfo", "r", encoding="utf-8") as _mem:
            for _line in _mem:
                if _line.startswith("MemTotal:"):
                    return int(_line.split()[1]) / (1024**2)
    except (OSError, ValueError, IndexError):
        pass
    return 8.0  # conservative fallback when the OS does not expose RAM details

TOTAL_RAM_GIB = _detect_total_ram_gib()

# Explicit user-selected concurrency. This deliberately overrides the conservative
# RAM/CPU auto-detection and SENTINEL_SCENE_WORKERS environment variable.
# Monitor RAM, swap, and scene failure rates during the first batch.
SCENE_WORKERS = 8
OPTIONAL_MISSING_SCENE_IDS = {
    "S1B_IW_GRDH_1SDV_20190120T064614_20190120T064639_014574_01B268_rtc",
}
SEARCH_WORKERS = max(1, min(4, int(os.environ.get("SENTINEL_SEARCH_WORKERS", 4))))
# Keep per-scene polarization reads serial; scene-level concurrency already overlaps I/O.
PARALLEL_POLARIZATION_READS = False
POLARIZATION_READ_WORKERS = 1
YEAR_WORKERS = 1
SEARCH_RETRIES = 3
STAC_REQUEST_TIMEOUT_SECONDS = 45
READ_RETRIES = 2
SCENE_RETRIES = 2
PC_TOKEN_REFRESH_MARGIN_S = 600
PC_TOKEN_CONNECT_TIMEOUT_SECONDS = 5
PC_TOKEN_READ_TIMEOUT_SECONDS = 10
PC_TOKEN_RETRIES = 3
PC_TOKEN_COOLDOWN_SECONDS = 30
SCENE_FAILURE_STREAK_LIMIT = 3
# Keep metadata discovery independent of high-latency raster reads.
STREAM_SCENES_DURING_INVENTORY = False

# ------------------------------------------------------------
# GDAL HTTP settings
# ------------------------------------------------------------
GDAL_ENV = dict(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff,.TIF,.TIFF",
    CPL_VSIL_CURL_USE_HEAD="NO",
    GDAL_INGESTED_BYTES_AT_OPEN=str(256 * 1024),
    GDAL_HTTP_MAX_RETRY="1",
    GDAL_HTTP_RETRY_DELAY="1",
    GDAL_HTTP_RETRY_CODES="429,500,502,503,504",
    GDAL_HTTP_TIMEOUT="30",
    # Use HTTP/2 over TLS when the installed GDAL/libcurl supports it.
    GDAL_HTTP_VERSION="2TLS",
    GDAL_HTTP_CONNECTTIMEOUT="10",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MULTIPLEX="YES",
    VSI_CACHE="TRUE",
    VSI_CACHE_SIZE=str(2 * 1024 * 1024),
    GDAL_CACHEMAX=32,
)

print("Existing output namespace (resume):", OUTPUT_DIR)
print("Legacy STAC cache:", LEGACY_STAC_CACHE_DIR)
print("New STAC cache:", STAC_CACHE_DIR)
print("Scene inventory DB:", SCENE_INDEX_DB_PATH)
print("Scene observation shards:", SCENE_OBS_DIR)
print("Final output:", FINAL_OUTPUT_DIR)
print("Patch:", f"{PATCH_SIZE} x {PATCH_SIZE} @ {MODEL_RESOLUTION_M} m")
print("Source resolution:", S1_RESOLUTION_M, "m")
print("Nearest-date tolerance:", MAX_NEAREST_DAYS, "days")
print("Stored bands:", OUTPUT_BANDS)
print("Remote polarizations to read:", READ_POLARIZATIONS)
print("Production raster read strategy:", PRODUCTION_READ_STRATEGY)
if PRODUCTION_READ_STRATEGY == "overview":
    print("Overview resampling:", "average; approximate versus exact native-pixel means")
print("STAC workers:", SEARCH_WORKERS)
print("GDAL HTTP version preference:", GDAL_ENV["GDAL_HTTP_VERSION"])
print("GDAL per-request timeout:", GDAL_ENV["GDAL_HTTP_TIMEOUT"], "seconds")
print("Streaming scenes during inventory:", STREAM_SCENES_DURING_INVENTORY)
print("Scene workers:", SCENE_WORKERS)
print("Parallel polarization reads:", PARALLEL_POLARIZATION_READS)
print("Max simultaneous polarization assets per scene:", POLARIZATION_READ_WORKERS)
print("COG tile model:", f"{COG_BLOCK_PX} x {COG_BLOCK_PX} ({COG_BLOCK_M:.2f} km)")


Existing output namespace (resume): Data/Global/pipeline_stage1/sentinel1
Legacy STAC cache: Data/Global/pipeline_stage1/sentinel1/stac_cache_v4
New STAC cache: Data/Global/pipeline_stage1/sentinel1/stac_cache_nearest_v1
Scene inventory DB: Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
Scene observation shards: Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Final output: Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1
Patch: 13 x 13 @ 1000 m
Source resolution: 10 m
Nearest-date tolerance: 18 days
Stored bands: ['vv', 'vh', 'hh', 'hv', 'vv_vh_ratio']
Remote polarizations to read: ('vv', 'vh')
Production raster read strategy: overview
Overview resampling: average; approximate versus exact native-pixel means
STAC workers: 4
GDAL HTTP version preference: 2TLS
GDAL per-request timeout: 30 seconds
Streaming scenes during inventory: False
Scene workers: 8
Parallel polarization reads: False
Max simultaneous polarization as

In [25]:
# ============================================================
# CONFIGURATION SANITY CHECKS
# ============================================================

assert SCENE_OBS_DIR.exists()
assert SCENE_INDEX_DB_PATH.parent.exists()
assert FINAL_OUTPUT_DIR.exists()
assert REQUIRED_POLARIZATIONS == ("vv", "vh")
assert READ_POLARIZATIONS == ("vv", "vh")
assert all(pol in REAL_BANDS for pol in REQUIRED_POLARIZATIONS)
assert SCENE_WORKERS >= 1
assert SEARCH_WORKERS >= 1
assert POLARIZATION_READ_WORKERS >= 1
assert isinstance(GDAL_ENV.get("GDAL_CACHEMAX"), int)
assert STAC_REQUEST_TIMEOUT_SECONDS > 0
assert MAX_NEAREST_DAYS >= 0
assert len(OUTPUT_BANDS) == 5

print("Configuration OK")
print("Scene shards:", SCENE_OBS_DIR)
print("Scene inventory DB:", SCENE_INDEX_DB_PATH)
print("STAC search workers:", SEARCH_WORKERS)
print("Scene read workers:", SCENE_WORKERS)
print("GDAL cache max:", GDAL_ENV["GDAL_CACHEMAX"], "MB")
print("STAC request timeout:", STAC_REQUEST_TIMEOUT_SECONDS, "seconds")
print("Nearest-date tolerance:", MAX_NEAREST_DAYS, "days")


Configuration OK
Scene shards: Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Scene inventory DB: Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
STAC search workers: 4
Scene read workers: 8
GDAL cache max: 32 MB
STAC request timeout: 45 seconds
Nearest-date tolerance: 18 days


### Production settings

The scene inventory is stored incrementally in SQLite, not by repeatedly rewriting one giant JSON index. For each spatial cluster, STAC searches are limited to the **union of the date intervals actually needed by the samples** (sample date ± `MAX_NEAREST_DAYS`), rather than querying every day between the cluster's first and last sample date. Overlapping intervals are merged, the query result for each interval/polarization pair is cached independently, and completed cluster results are checkpointed. Newly discovered scenes enter a bounded remote-COG worker queue while inventory continues; global station-patch indexing ensures each scene shard includes relevant stations regardless of which cluster discovered the scene first. This lets an interrupted run resume without repeating successful requests.

For each station/sample date, output uses the nearest acquisition with valid data on either side of the target date; an exact distance tie prefers the earlier acquisition. There is no temporal median. The current safety tolerance is configurable as `MAX_NEAREST_DAYS` (18 days by default); no observation inside that tolerance leaves the sample as NaN. Chosen acquisition offsets are stored in the Zarr output.


**Fast VV/VH-only mode:** the production scene reader downloads only VV and VH. HH/HV are excluded before any remote raster open. The original four-band checkpoint/output layout is retained so existing v9 scene shards stay compatible; pre-existing cached HH/HV values are not deleted, while newly read scene shards leave those two slots empty.


**Low-memory Azure profile:** the notebook is configured for one search worker and one scene worker, disables concurrent polarization reads, uses a smaller GDAL/VSI cache, and reads only the input columns needed downstream. This is intentionally conservative for a 2-vCPU / 4-GB VM.


## Planetary Computer metadata + remote COG range processing

The STAC metadata API is public. Azure raster assets are read using a Planetary Computer SAS token; no subscription/API key is required by this notebook.

The data path is:

1. For each cluster, merge the date windows surrounding actual sample dates; do not query the empty gaps between widely separated target dates.
2. Cache each STAC interval/polarization query and commit completed cluster results incrementally to SQLite.
3. Start bounded remote COG reads while inventory searches continue; a global station-patch spatial index keeps shared-scene extraction complete.
4. Read only native, tile-aligned 10 m COG windows via HTTP range requests.
5. Aggregate locally to 1 km and select the single nearest valid acquisition for each sample/polarization pair (earlier date wins ties).
6. Store scene observations and final yearly Zarr arrays so interrupted runs can resume.

`MAX_NEAREST_DAYS` defines the maximum allowed distance from a target date, not a rolling average/median window. Increase it if a wider nearest-date search is intended.


### HTTP behavior and recovery

STAC metadata searches use GET for the small bbox/date queries, with bounded transport retries and application-level backoff. Raster COG reads request HTTP/2 over TLS when the installed GDAL/libcurl supports it; GDAL's HTTP/2 multiplexing can improve parallel byte-range reads. If the local GDAL/libcurl build reports HTTP-version/protocol errors, change `GDAL_HTTP_VERSION` to `"1.1"`. HTTP/2 does not fix a machine/network outage; network-unreachable errors are retried with backoff.


In [26]:
def _clean_href(href):
    """Normalize STAC asset hrefs before appending a fresh SAS token."""
    if href is None:
        raise ValueError("Asset href is None")
    href = str(href).strip()
    if href.startswith("[") and "](" in href and href.endswith(")"):
        href = href[1:href.index("](")]
    href = href.split("?", 1)[0]
    return href


class PCTokenManager:
    """Thread-safe, auto-refreshing, anonymous SAS token for one PC collection."""

    def __init__(self, collection, sas_url=PC_SAS_URL, margin_s=PC_TOKEN_REFRESH_MARGIN_S):
        self.collection = collection
        self.sas_url = sas_url.rstrip("/")
        self.margin_s = margin_s
        self._lock = threading.Lock()
        self._session = requests.Session()
        self._token = None
        self.expiry = None
        self.lifetime_s = None
        self._retry_after_monotonic = 0.0
        self._last_error = None

    def seconds_left(self) -> float:
        if self.expiry is None:
            return -1.0
        return (self.expiry - pd.Timestamp.now(tz="UTC")).total_seconds()

    def _margin(self) -> float:
        # Never let the margin exceed a quarter of the token lifetime, otherwise a
        # short-lived token would be re-minted on every single call.
        if self.lifetime_s:
            return min(self.margin_s, 0.25 * self.lifetime_s)
        return self.margin_s

    @staticmethod
    def _retry_delay(resp, attempt) -> float:
        try:
            return min(float(resp.headers["Retry-After"]), 30.0)
        except (KeyError, ValueError):
            return float(min(2 ** attempt, 30))

    def _fetch(self):
        now = time.monotonic()
        if now < self._retry_after_monotonic:
            raise requests.ConnectionError(
                "Planetary Computer token endpoint is in a short cooldown after a recent failure: "
                f"{self._last_error or 'connection failure'}"
            )

        url = f"{self.sas_url}/{self.collection}"
        last_error = None
        for attempt in range(PC_TOKEN_RETRIES):
            resp = None
            try:
                resp = self._session.get(
                    url,
                    timeout=(PC_TOKEN_CONNECT_TIMEOUT_SECONDS, PC_TOKEN_READ_TIMEOUT_SECONDS),
                )
                if resp.status_code in (429, 500, 502, 503, 504) and attempt < PC_TOKEN_RETRIES - 1:
                    delay = self._retry_delay(resp, attempt)
                    resp.close()
                    time.sleep(min(delay, 4.0))
                    continue
                resp.raise_for_status()
                body = resp.json()
                expiry = pd.Timestamp(body["msft:expiry"])
                self.expiry = (
                    expiry.tz_localize("UTC") if expiry.tzinfo is None
                    else expiry.tz_convert("UTC")
                )
                self._token = body["token"]
                self.lifetime_s = self.seconds_left()
                self._retry_after_monotonic = 0.0
                self._last_error = None
                return
            except requests.RequestException as exc:
                last_error = exc
                if resp is not None:
                    resp.close()
                if attempt >= PC_TOKEN_RETRIES - 1:
                    self._last_error = f"{type(exc).__name__}: {exc}"
                    self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
                    raise
                delay = min(4.0, 2 ** attempt)
                print(
                    f"Planetary Computer token request failed "
                    f"(attempt {attempt + 1}/{PC_TOKEN_RETRIES}): "
                    f"{type(exc).__name__}: {exc}; retrying in {delay:.0f}s."
                )
                time.sleep(delay)
            except (KeyError, ValueError, TypeError) as exc:
                if resp is not None:
                    resp.close()
                self._last_error = f"invalid token response: {exc}"
                self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
                raise RuntimeError(f"Invalid Planetary Computer SAS token response: {exc}") from exc
        if last_error is not None:
            self._last_error = f"{type(last_error).__name__}: {last_error}"
            self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
            raise last_error
        raise requests.ConnectionError("Planetary Computer SAS token request did not complete")

    def token(self, force: bool = False) -> str:
        with self._lock:
            if force or self._token is None or self.seconds_left() < self._margin():
                self._fetch()
            return self._token

    def sign(
            self,
            href: str,
            force: bool = False,
        ) -> str:

            href = _clean_href(href)

            return (
                f"{href}?{self.token(force)}"
            )


rtc_token = PCTokenManager(S1_COLLECTION)
print("SAS token manager configured; token is fetched on demand during scene extraction.")


SAS token manager configured; token is fetched on demand during scene extraction.


In [27]:
# ============================================================
# LOW-MEMORY INPUT LOAD + VALIDATION
# ============================================================

SAMPLES_PATH = Path("Data/Global/pipeline_stage1/station_date_candidates.parquet")
STATIONS_PATH = Path("Data/Global/pipeline_stage1/stations.parquet")
SAMPLE_REQUIRED_COLUMNS = {"station_id", "sample_date"}
STATION_REQUIRED_COLUMNS = {"station_id", "latitude", "longitude"}


def _memory_mib(df):
    """Approximate DataFrame footprint, including Python objects where possible."""
    return float(df.memory_usage(index=True, deep=True).sum()) / (1024 ** 2)


def _parquet_column_names(path):
    """Read only Parquet schema metadata, not its column data."""
    try:
        import pyarrow.parquet as _pq
        return list(_pq.ParquetFile(path).schema.names)
    except ImportError:
        try:
            import fastparquet as _fp
            return list(_fp.ParquetFile(path).columns)
        except ImportError as exc:
            raise RuntimeError(
                "Memory-safe projected Parquet loading needs a Parquet engine "
                "(PyArrow or fastparquet). Install one in this environment and rerun this cell."
            ) from exc


def _load_projected_frame(path, required, optional=()):
    if not path.exists():
        raise FileNotFoundError(f"Required input Parquet file not found: {path}")
    columns_available = _parquet_column_names(path)
    missing = set(required) - set(columns_available)
    if missing:
        raise KeyError(f"{path} is missing required columns: {sorted(missing)}")
    # Read only fields that are used downstream; never materialize unused columns.
    columns = [c for c in (*optional, *required) if c in columns_available]
    frame = pd.read_parquet(path, columns=columns)
    return frame, columns_available


# If this cell is rerun in an existing kernel, trim any already-loaded wide frames.
# A fresh kernel is recommended after an OOM so stale objects have been released.
if "samples" not in globals():
    samples, _sample_source_columns = _load_projected_frame(
        SAMPLES_PATH,
        required=("station_id", "sample_date"),
        optional=("sample_id",),
    )
else:
    _sample_keep = [c for c in ("sample_id", "station_id", "sample_date") if c in samples.columns]
    _sample_missing = SAMPLE_REQUIRED_COLUMNS - set(_sample_keep)
    if _sample_missing:
        raise KeyError(f"samples is missing required columns: {sorted(_sample_missing)}")
    samples = samples.loc[:, _sample_keep]

if "stations" not in globals():
    stations, _station_source_columns = _load_projected_frame(
        STATIONS_PATH,
        required=("station_id", "latitude", "longitude"),
    )
else:
    _station_keep = ["station_id", "latitude", "longitude"]
    _station_missing = STATION_REQUIRED_COLUMNS - set(stations.columns)
    if _station_missing:
        raise KeyError(f"stations is missing required columns: {sorted(_station_missing)}")
    stations = stations.loc[:, _station_keep]

for name, df, required in [
    ("samples", samples, SAMPLE_REQUIRED_COLUMNS),
    ("stations", stations, STATION_REQUIRED_COLUMNS),
]:
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{name} is missing columns: {sorted(missing)}")

# Normalize types in-place on these narrowly projected tables.
samples["sample_date"] = pd.to_datetime(samples["sample_date"], errors="coerce").dt.normalize()
samples["station_id"] = samples["station_id"].astype(str)
stations["station_id"] = stations["station_id"].astype(str)

# ---- hygiene: bad coordinates, duplicates, stations nobody needs ----
n_st0, n_sm0 = len(stations), len(samples)

stations = stations.dropna(subset=["latitude", "longitude"])
stations = stations.loc[
    stations["latitude"].between(-90, 90)
    & stations["longitude"].between(-180, 180)
]
stations = stations.drop_duplicates("station_id").reset_index(drop=True)

samples = samples.dropna(subset=["sample_date"])
samples = samples.loc[samples["station_id"].isin(stations["station_id"])]

# Remove unused station rows, avoiding a merge-sized intermediate table.
sample_station_ids = pd.Index(samples["station_id"].unique())
stations = stations.loc[stations["station_id"].isin(sample_station_ids)].reset_index(drop=True)
samples = samples.reset_index(drop=True)
del sample_station_ids
gc.collect()

try:
    import psutil as _psutil
    _available_gib = _psutil.virtual_memory().available / (1024 ** 3)
    print(f"System RAM currently available: {_available_gib:.2f} GiB")
except Exception:
    pass

print(f"Stations: {len(stations):,} (dropped {n_st0 - len(stations):,}) | {_memory_mib(stations):,.1f} MiB")
print(f"Samples:  {len(samples):,} (dropped {n_sm0 - len(samples):,}) | {_memory_mib(samples):,.1f} MiB")
print("Loaded columns only:", list(samples.columns), "and", list(stations.columns))
if not samples.empty:
    print("Date range:", samples["sample_date"].min(), "->", samples["sample_date"].max())
else:
    raise ValueError("No usable station-date samples remain after input validation.")

# Drop schema-only names/diagnostic references before the heavier geometry/index cells.
for _name in ("_sample_source_columns", "_station_source_columns", "_sample_keep", "_station_keep", "_sample_missing", "_station_missing", "_psutil", "_available_gib"):
    globals().pop(_name, None)
del _name
gc.collect()


System RAM currently available: 2.95 GiB
Stations: 2,210 (dropped 0) | 0.1 MiB
Samples:  3,514,643 (dropped 0) | 144.0 MiB
Loaded columns only: ['sample_id', 'station_id', 'sample_date'] and ['station_id', 'latitude', 'longitude']
Date range: 2015-04-01 00:00:00 -> 2026-10-03 00:00:00


0

In [28]:
# STAC search is public metadata; production workers create their own client
# per thread so one HTTP client is never shared concurrently across workers.
catalog = pystac_client.Client.open(_normalize_pc_stac_url(PC_STAC_URL))
_ = catalog.get_collection(S1_COLLECTION)

print("Planetary Computer STAC connected:", S1_COLLECTION)


Planetary Computer STAC connected: sentinel-1-rtc


In [29]:
# ============================================================
# GEOMETRY + SAMPLE-DATE INDEX
# ============================================================

EARTH_R_KM = 6371.0088
EPOCH = pd.Timestamp("1970-01-01")


def ground_xy_km(
    lon,
    lat,
):
    """
    Approximate local metric coordinates in km.

    Used only for the storage-aware station clustering.
    Actual raster geometry is always handled in the scene's
    own projected CRS.
    """

    lat_r = np.radians(lat)

    return (
        EARTH_R_KM
        * np.radians(lon)
        * np.cos(lat_r),

        EARTH_R_KM
        * lat_r,
    )


def station_patch_boxes_deg(
    df,
):
    """
    Approximate 13 km x 13 km station footprints in lon/lat.

    Used for STAC footprint filtering only.

    Actual Sentinel-1 reads are constructed in the scene's
    projected CRS from exact pixel geometry.
    """

    lon = df[
        "longitude"
    ].to_numpy(float)

    lat = df[
        "latitude"
    ].to_numpy(float)

    radius_km = (
        PATCH_RADIUS_M / 1000.0
    )

    dlat = np.degrees(
        radius_km / EARTH_R_KM
    )

    lat_edge = np.minimum(
        np.abs(lat) + dlat,
        89.0,
    )

    dlon = np.degrees(
        radius_km
        /
        (
            EARTH_R_KM
            *
            np.cos(
                np.radians(
                    lat_edge
                )
            )
        )
    )

    return np.column_stack(
        [
            np.clip(
                lon - dlon,
                -180,
                180,
            ),

            np.clip(
                lat - dlat,
                -90,
                90,
            ),

            np.clip(
                lon + dlon,
                -180,
                180,
            ),

            np.clip(
                lat + dlat,
                -90,
                90,
            ),
        ]
    )


def sample_days_by_station(
    samples_df,
):
    """
    Sorted unique sample days per station.
    """

    result = {}

    for (
        station_id,
        group,
    ) in samples_df.groupby(
        "station_id"
    ):

        days = (
            group[
                "sample_date"
            ]
            .to_numpy(
                dtype="datetime64[D]"
            )
            .astype(
                np.int64
            )
        )

        result[
            station_id
        ] = np.unique(
            days
        )

    return result


# Full global sample-date index.
SAMPLE_DAYS = (
    sample_days_by_station(
        samples
    )
)


print(
    "COG tile model:",
    f"{COG_BLOCK_PX} x {COG_BLOCK_PX}",
)

print(
    "Native patch:",
    f"{SOURCE_PATCH_PX} x {SOURCE_PATCH_PX}",
)

print(
    "Native pixels / model cell:",
    SOURCE_PIXELS_PER_MODEL_CELL,
)

COG tile model: 512 x 512
Native patch: 1300 x 1300
Native pixels / model cell: 100


## Clustering

Recursive bisection in ground-km space with a storage-aware cut rule.

The weighted graph favors keeping stations together when their 13 km patches are likely to share 512×512 RTC COG tiles and their sample dates overlap within the nearest-acquisition tolerance. The hard cluster diameter/station limits cap memory and prevent long chain-shaped clusters.


In [30]:
# ============================================================
# STORAGE-AWARE CLUSTERING
# ============================================================

def temporal_overlap_score(
    days_a,
    days_b,
    window_days=S1_WINDOW_DAYS,
):
    """
    Approximate number of source windows that can share
    Sentinel-1 acquisitions.

    We don't count every possible date pair. For every sample
    day at A, we only ask whether B has at least one sample
    within +/- window_days, and vice versa.
    """

    if (
        len(days_a) == 0
        or len(days_b) == 0
    ):
        return 0.0

    a = np.asarray(
        days_a,
        dtype=np.int64,
    )

    b = np.asarray(
        days_b,
        dtype=np.int64,
    )

    def one_way(
        x,
        y,
    ):
        k = np.searchsorted(
            y,
            x,
            side="left",
        )

        k0 = np.clip(
            k - 1,
            0,
            len(y) - 1,
        )

        k1 = np.clip(
            k,
            0,
            len(y) - 1,
        )

        distance = np.minimum(
            np.abs(
                x - y[k0]
            ),
            np.abs(
                x - y[k1]
            ),
        )

        return (
            distance
            <= window_days
        )

    return 0.5 * (
        one_way(a, b).sum()
        +
        one_way(b, a).sum()
    )


def tile_box_for_station(
    x,
    y,
    phase_x,
    phase_y,
):
    """
    Approximate source COG tiles touched by one station patch.

    We average across several unknown tile-grid phases because
    each RTC scene's projected grid origin can differ.
    """

    block = COG_BLOCK_M

    r = PATCH_RADIUS_M / 1000

    return np.array(
        [
            math.floor(
                (x - r - phase_x)
                / block
            ),

            math.floor(
                (y - r - phase_y)
                / block
            ),

            math.ceil(
                (x + r - phase_x)
                / block
            ),

            math.ceil(
                (y + r - phase_y)
                / block
            ),
        ],
        dtype=np.int64,
    )


def expected_shared_tile_savings(
    x1,
    y1,
    x2,
    y2,
):
    """
    Expected number of internal COG tiles saved when two
    station windows are read together rather than separately.
    """

    phases = [
        0.00,
        0.25 * COG_BLOCK_M,
        0.50 * COG_BLOCK_M,
        0.75 * COG_BLOCK_M,
    ]

    values = []

    for phase_x in phases:

        for phase_y in phases:

            a = tile_box_for_station(
                x1,
                y1,
                phase_x,
                phase_y,
            )

            b = tile_box_for_station(
                x2,
                y2,
                phase_x,
                phase_y,
            )

            area_a = (
                a[2] - a[0]
            ) * (
                a[3] - a[1]
            )

            area_b = (
                b[2] - b[0]
            ) * (
                b[3] - b[1]
            )

            union = (
                max(
                    a[2],
                    b[2],
                )
                -
                min(
                    a[0],
                    b[0],
                )
            ) * (
                max(
                    a[3],
                    b[3],
                )
                -
                min(
                    a[1],
                    b[1],
                )
            )

            values.append(
                max(
                    0,
                    area_a
                    + area_b
                    - union,
                )
            )

    return float(
        np.mean(values)
    )


def build_storage_edges(
    stations_df,
):
    """
    Build a weighted graph.

    Edge weight approximates:

        shared COG tiles
        ×
        temporal co-occurrence

    Only geographically plausible pairs are considered.
    """

    x, y = ground_xy_km(
        stations_df[
            "longitude"
        ].to_numpy(float),

        stations_df[
            "latitude"
        ].to_numpy(float),
    )

    coords = np.column_stack(
        [x, y]
    )

    share_distance_km = (
        2
        * PATCH_RADIUS_M
        / 1000
        + COG_BLOCK_M
    )

    pairs = (
        cKDTree(
            coords
        )
        .query_pairs(
            share_distance_km,
            p=np.inf,
            output_type="ndarray",
        )
    )

    station_ids = (
        stations_df[
            "station_id"
        ].to_numpy()
    )

    edges = []

    for i, j in pairs:

        spatial = (
            expected_shared_tile_savings(
                x[i],
                y[i],
                x[j],
                y[j],
            )
        )

        if spatial <= 0:
            continue

        temporal = (
            temporal_overlap_score(
                SAMPLE_DAYS[
                    station_ids[i]
                ],

                SAMPLE_DAYS[
                    station_ids[j]
                ],
            )
        )

        if temporal <= 0:
            continue

        edges.append(
            (
                int(i),
                int(j),
                spatial * temporal,
            )
        )

    if not edges:
        return np.empty(
            (0, 3),
            dtype=np.float64,
        )

    return np.asarray(
        edges,
        dtype=np.float64,
    )


def build_station_clusters(
    stations_df,
):
    """
    Recursive bisection.

    A cut is preferred when it severs the smallest amount of
    expected Azure tile reuse.

    Only the hard side/station limits force a split.
    """

    x, y = ground_xy_km(
        stations_df[
            "longitude"
        ].to_numpy(float),

        stations_df[
            "latitude"
        ].to_numpy(float),
    )

    n = len(
        stations_df
    )

    edges = build_storage_edges(
        stations_df
    )

    labels = np.full(
        n,
        -1,
        dtype=np.int64,
    )

    stack = [
        np.arange(
            n,
            dtype=np.int64,
        )
    ]

    next_cluster = 0

    while stack:

        idx = stack.pop()

        width = np.ptp(
            x[idx]
        )

        height = np.ptp(
            y[idx]
        )

        if (
            len(idx)
            <= MAX_CLUSTER_STATIONS
            and
            width
            <= MAX_CLUSTER_SIDE_KM
            and
            height
            <= MAX_CLUSTER_SIDE_KM
        ):

            labels[idx] = (
                next_cluster
            )

            next_cluster += 1
            continue

        axis = (
            x
            if width >= height
            else y
        )

        ordered = idx[
            np.argsort(
                axis[idx],
                kind="stable",
            )
        ]

        values = axis[
            ordered
        ]

        m = len(
            ordered
        )

        lo = max(
            1,
            m // 4,
        )

        hi = min(
            m - 1,
            m - m // 4,
        )

        gaps = np.diff(
            values
        )[
            lo - 1:hi
        ]

        # Global station -> local order.
        position = np.full(
            n,
            -1,
            dtype=np.int64,
        )

        position[
            ordered
        ] = np.arange(
            m
        )

        if len(edges):

            e0 = edges[
                :, 0
            ].astype(
                np.int64
            )

            e1 = edges[
                :, 1
            ].astype(
                np.int64
            )

            inside = (
                (position[e0] >= 0)
                &
                (position[e1] >= 0)
            )

            a = position[
                e0[inside]
            ]

            b = position[
                e1[inside]
            ]

            weights = edges[
                inside,
                2,
            ]

            left = np.minimum(
                a,
                b,
            )

            right = np.maximum(
                a,
                b,
            )

            diff = np.zeros(
                m + 1,
                dtype=np.float64,
            )

            np.add.at(
                diff,
                left + 1,
                weights,
            )

            np.add.at(
                diff,
                right + 1,
                -weights,
            )

            severed = np.cumsum(
                diff
            )[
                lo:hi + 1
            ]

        else:

            severed = np.zeros(
                hi - lo + 1,
                dtype=np.float64,
            )

        best = np.flatnonzero(
            severed
            == severed.min()
        )

        # After minimizing Azure tile reuse severed,
        # use the widest geographic gap as a tie-break.
        if (
            gaps[
                best
            ].max()
            > 0
        ):

            k = (
                lo
                +
                int(
                    best[
                        np.argmax(
                            gaps[
                                best
                            ]
                        )
                    ]
                )
            )

        else:

            k = (
                lo
                +
                int(
                    best[
                        np.argmin(
                            np.abs(
                                lo
                                + best
                                - m / 2
                            )
                        )
                    ]
                )
            )

        stack.append(
            ordered[:k]
        )

        stack.append(
            ordered[k:]
        )

    return pd.DataFrame(
        {
            "station_id":
                stations_df[
                    "station_id"
                ].to_numpy(),

            "cluster_id":
                labels,
        }
    )


# Reuse the exact station-to-cluster labels that were used to create the SQLite inventory.
# Recomputing this storage-aware graph can change labels on a tie and would make
# scene_clusters refer to the wrong stations. This resume notebook fails closed if
# the persisted mapping is missing or incompatible.
_existing_mapping_path = OUTPUT_DIR / "stations_with_clusters.parquet"
if not _existing_mapping_path.exists():
    raise FileNotFoundError(
        f"Required saved cluster mapping is missing: {_existing_mapping_path}. "
        "Copy the existing stations_with_clusters.parquet from the local project first."
    )

_saved_mapping = pd.read_parquet(_existing_mapping_path)
if not {"station_id", "cluster_id"}.issubset(_saved_mapping.columns):
    raise RuntimeError(
        f"Saved cluster mapping has unexpected columns: {list(_saved_mapping.columns)}"
    )
_saved_mapping = _saved_mapping[["station_id", "cluster_id"]].copy()
_saved_mapping["station_id"] = _saved_mapping["station_id"].astype(str)
_saved_mapping["cluster_id"] = pd.to_numeric(
    _saved_mapping["cluster_id"], errors="raise"
).astype(np.int64)
if _saved_mapping["station_id"].duplicated().any():
    raise RuntimeError("Saved cluster mapping contains duplicate station IDs.")

_current_station_ids = set(stations["station_id"].astype(str))
_saved_station_ids = set(_saved_mapping["station_id"])
if _current_station_ids != _saved_station_ids:
    missing = len(_current_station_ids - _saved_station_ids)
    extra = len(_saved_station_ids - _current_station_ids)
    raise RuntimeError(
        "Saved cluster mapping does not match stations.parquet "
        f"(missing station IDs={missing:,}; extra station IDs={extra:,})."
    )

station_clusters = _saved_mapping
stations_cluster = stations.merge(
    station_clusters,
    on="station_id",
    how="left",
    validate="one_to_one",
)
if stations_cluster["cluster_id"].isna().any():
    raise RuntimeError("Saved cluster mapping leaves stations without a cluster ID.")

if {"latitude", "longitude"}.issubset(pd.read_parquet(_existing_mapping_path).columns):
    _saved_coords = pd.read_parquet(
        _existing_mapping_path, columns=["station_id", "latitude", "longitude"]
    ).copy()
    _saved_coords["station_id"] = _saved_coords["station_id"].astype(str)
    _coord_check = stations[["station_id", "latitude", "longitude"]].merge(
        _saved_coords, on="station_id", suffixes=("_current", "_saved"),
        validate="one_to_one"
    )
    for _coord in ("latitude", "longitude"):
        if not np.allclose(
            _coord_check[f"{_coord}_current"].to_numpy(float),
            _coord_check[f"{_coord}_saved"].to_numpy(float),
            rtol=0, atol=1e-8, equal_nan=True,
        ):
            raise RuntimeError(
                f"Saved cluster map coordinates disagree with stations.parquet ({_coord})."
            )

print(
    "Reusing saved station-cluster map:",
    f"{stations_cluster['station_id'].nunique():,} stations / "
    f"{stations_cluster['cluster_id'].nunique():,} clusters"
)
assert (stations_cluster["cluster_id"] >= 0).all()

# Global spatial index for safe inventory/extraction overlap. It is built once
# and avoids scanning every station for every scene. STRtree query results are
# station-row positions in GLOBAL_STATION_FRAME.
GLOBAL_STATION_FRAME = stations_cluster[
    ["station_id", "latitude", "longitude", "cluster_id"]
].copy().reset_index(drop=True)
_GLOBAL_PATCH_BOXES = station_patch_boxes_deg(GLOBAL_STATION_FRAME)
GLOBAL_STATION_PATCH_GEOMETRIES = shapely.box(
    _GLOBAL_PATCH_BOXES[:, 0],
    _GLOBAL_PATCH_BOXES[:, 1],
    _GLOBAL_PATCH_BOXES[:, 2],
    _GLOBAL_PATCH_BOXES[:, 3],
)
GLOBAL_STATION_TREE = STRtree(GLOBAL_STATION_PATCH_GEOMETRIES)
print("Global station-patch spatial index:", len(GLOBAL_STATION_FRAME), "stations")

print(
    "Clusters:",
    stations_cluster[
        "cluster_id"
    ].nunique(),
)

print(
    "Largest cluster:",
    stations_cluster.groupby(
        "cluster_id"
    ).size().max(),
)

Reusing saved station-cluster map: 2,210 stations / 819 clusters
Global station-patch spatial index: 2210 stations
Clusters: 819
Largest cluster: 50


In [31]:
# ============================================================
# CLUSTER TABLE
# ============================================================

def make_cluster_table(
    stations_df,
    samples_df,
    station_cluster_lookup,
):
    s = stations_df[
        [
            "station_id",
            "cluster_id",
            "latitude",
            "longitude",
        ]
    ].copy()

    boxes = (
        station_patch_boxes_deg(
            s
        )
    )

    s[
        "b0"
    ], s[
        "b1"
    ], s[
        "b2"
    ], s[
        "b3"
    ] = boxes.T

    x, y = ground_xy_km(
        s["longitude"].to_numpy(float),
        s["latitude"].to_numpy(float),
    )

    s["x"] = x
    s["y"] = y

    g = (
        s.groupby(
            "cluster_id"
        )
        .agg(
            bbox_min_lon=("b0", "min"),
            bbox_min_lat=("b1", "min"),
            bbox_max_lon=("b2", "max"),
            bbox_max_lat=("b3", "max"),

            n_stations=(
                "station_id",
                "size",
            ),

            x0=("x", "min"),
            x1=("x", "max"),
            y0=("y", "min"),
            y1=("y", "max"),
        )
    )

    g[
        "width_km"
    ] = (
        g["x1"]
        - g["x0"]
        + PATCH_WIDTH_M / 1000
    )

    g[
        "height_km"
    ] = (
        g["y1"]
        - g["y0"]
        + PATCH_WIDTH_M / 1000
    )

    # Mapping + Series groupby avoids materializing a sample-sized merge table.
    sample_cluster_ids = samples_df["station_id"].map(station_cluster_lookup)
    if sample_cluster_ids.isna().any():
        raise RuntimeError("Some sample station IDs have no cluster assignment.")
    d = samples_df["sample_date"].groupby(sample_cluster_ids, sort=False).agg(
        min_date="min",
        max_date="max",
        n_samples="size",
    )
    d.index.name = "cluster_id"
    del sample_cluster_ids

    return (
        g
        .drop(
            columns=[
                "x0",
                "x1",
                "y0",
                "y1",
            ]
        )

        .join(d)

        .reset_index()
    )


station_cluster_lookup = stations_cluster.set_index("station_id")["cluster_id"]
if not station_cluster_lookup.index.is_unique:
    raise RuntimeError("Station-to-cluster mapping is not unique; refusing an ambiguous lookup.")

clusters = (
    make_cluster_table(
        stations_cluster,
        samples,
        station_cluster_lookup,
    )
    .sort_values(
        "cluster_id"
    )
    .reset_index(
        drop=True
    )
)

# Build the small downstream sample table through a mapping, not a many-to-one merge.
_sample_columns = [c for c in ("sample_id", "station_id", "sample_date") if c in samples.columns]
samples_c = samples.loc[:, _sample_columns].copy()
_sample_cluster_ids = samples_c["station_id"].map(station_cluster_lookup)
if _sample_cluster_ids.isna().any():
    raise RuntimeError("Some samples could not be assigned to a station cluster.")
samples_c["cluster_id"] = _sample_cluster_ids.astype("int32")
del _sample_cluster_ids, _sample_columns
gc.collect()

samples_c["year"] = samples_c["sample_date"].dt.year.astype("int16")

clusters.to_parquet(
    OUTPUT_DIR / "clusters.parquet",
    index=False,
)

stations_cluster.to_parquet(
    OUTPUT_DIR
    / "stations_with_clusters.parquet",
    index=False,
)

print(
    "Clusters:",
    len(clusters),
)

display(
    clusters.head(20)
)

Clusters: 819


,cluster_id,bbox_min_lon,bbox_min_lat,bbox_max_lon,bbox_max_lat,n_stations,width_km,height_km,min_date,max_date,n_samples
0,0,127.588640,36.981764,127.736040,37.098906,18,13.049375,13.025575,2015-04-01,2019-12-10,22502
1,1,126.893367,37.231044,127.061034,37.352556,10,14.660379,13.511497,2015-08-31,2017-11-29,2406
2,2,126.876883,37.878624,127.025847,37.996146,19,13.119585,13.067829,2015-04-04,2019-12-04,20509
3,3,116.359200,41.515974,116.515620,41.632886,1,13.000000,13.000000,2018-09-12,2019-11-12,302
4,4,115.460668,41.492304,116.439707,42.013766,5,41.251026,57.983970,2018-09-08,2019-11-12,1487
5,5,115.602080,41.704334,116.255364,42.083556,12,18.340978,42.167581,2018-07-28,2022-11-24,5148
6,6,115.730577,42.046824,116.447015,42.459666,8,42.117303,45.905960,2018-07-25,2019-11-22,2457
7,7,115.524679,41.721614,116.264362,42.154416,12,50.882304,48.125414,2018-07-25,2019-11-22,3588
8,8,115.610656,42.199154,116.301959,42.478196,6,57.222177,31.028058,2018-08-20,2019-11-22,1673
9,9,102.842710,38.570704,102.993619,38.688426,10,13.116862,13.090068,2021-01-22,2022-12-21,6039


In [32]:
# ============================================================
# GLOBAL UNIQUE SCENE INVENTORY -- SQLITE, INCREMENTAL, RESUMABLE
# ============================================================


def cluster_cache_path(cluster_id):
    return STAC_CACHE_DIR / f"cluster_{int(cluster_id):05d}.json"


def legacy_cluster_cache_path(cluster_id):
    return LEGACY_STAC_CACHE_DIR / f"cluster_{int(cluster_id):05d}.json"


def stac_day(timestamp):
    return pd.Timestamp(timestamp).strftime("%Y-%m-%dT00:00:00Z")


def load_stac_cache(cache_path):
    if not cache_path.exists() or cache_path.stat().st_size == 0:
        return None
    try:
        with cache_path.open("r", encoding="utf-8") as f:
            raw = json.load(f)
        if not isinstance(raw, list):
            return None
        items = []
        for entry in raw:
            if not isinstance(entry, dict) or not entry.get("id"):
                return None
            items.append(pystac.Item.from_dict(entry))
        return items
    except (json.JSONDecodeError, ValueError, KeyError, TypeError, OSError):
        return None


def save_stac_cache(cache_path, items):
    """Atomic cluster cache write; interrupted runs cannot leave valid-looking partial JSON."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = Path(str(cache_path) + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        f.write("[")
        for i, item in enumerate(items):
            if i:
                f.write(",")
            json.dump(item.to_dict(), f, separators=(",", ":"))
        f.write("]")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, cache_path)


_STAC_TLS = threading.local()
_STAC_PROGRESS_EVENTS = None


def _emit_inventory_event(kind, **payload):
    """Send lightweight worker events to the main thread for live progress."""
    event_queue = globals().get("_STAC_PROGRESS_EVENTS")
    if event_queue is not None:
        event = {"kind": kind, "at": time.monotonic()}
        event.update(payload)
        event_queue.put(event)


def _get_worker_catalog():
    """Independent STAC client with bounded transport retries per search thread."""
    client = getattr(_STAC_TLS, "catalog", None)
    if client is None:
        retry = Retry(
            total=2,
            connect=2,
            read=2,
            status=2,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=None,
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        stac_io = StacApiIO()
        adapter = HTTPAdapter(
            max_retries=retry,
            pool_connections=6,
            pool_maxsize=6,
        )
        stac_io.session.mount("https://", adapter)
        stac_io.session.mount("http://", adapter)
        client = pystac_client.Client.open(
            _normalize_pc_stac_url(PC_STAC_URL),
            stac_io=stac_io,
            timeout=STAC_REQUEST_TIMEOUT_SECONDS,
        )
        _STAC_TLS.catalog = client
    return client


def _query_stac_window(bbox, start, end, cluster_id):
    """Retrieve one bounded date window with live item-count events.

    Polarizations are filtered locally after retrieval. One spatial/temporal
    request per window avoids duplicate VV/VH and HH/HV requests and also avoids
    relying on exact array equality for the STAC polarization property.
    """
    datetime_range = f"{stac_day(start)}/{stac_day(end)}"
    query_id = (
        f"{int(cluster_id)}:{datetime_range}:{threading.get_ident()}:{time.monotonic_ns()}"
    )
    _emit_inventory_event(
        "query_start",
        query_id=query_id,
        cluster_id=int(cluster_id),
        start=str(pd.Timestamp(start).date()),
        end=str(pd.Timestamp(end).date()),
        datetime_range=datetime_range,
    )

    items = []
    try:
        search = _get_worker_catalog().search(
            collections=[S1_COLLECTION],
            bbox=bbox,
            datetime=datetime_range,
            method="GET",
            limit=500,
            max_items=None,
            # Item bbox remains available for conservative local spatial filtering.
            fields={"exclude": ["geometry"]},
        )
        for count, item in enumerate(search.items(), start=1):
            items.append(item)
            # Reduce cross-thread event traffic while still showing pages flowing.
            if count == 1 or count % 25 == 0:
                _emit_inventory_event(
                    "query_items",
                    query_id=query_id,
                    cluster_id=int(cluster_id),
                    items_seen=count,
                )
        _emit_inventory_event(
            "query_end",
            query_id=query_id,
            cluster_id=int(cluster_id),
            success=True,
            items=len(items),
        )
        return items
    except Exception as exc:
        _emit_inventory_event(
            "query_end",
            query_id=query_id,
            cluster_id=int(cluster_id),
            success=False,
            items=len(items),
            error=f"{type(exc).__name__}: {exc}",
        )
        raise


def _query_window_with_retry(bbox, start, end, cluster_id):
    """Retry bounded STAC windows with exponential backoff and visible status."""
    for attempt in range(SEARCH_RETRIES):
        try:
            return _query_stac_window(bbox, start, end, cluster_id)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if attempt >= SEARCH_RETRIES - 1:
                raise
            delay = min(60, 2 ** (attempt + 1))
            _emit_inventory_event(
                "query_retry",
                cluster_id=int(cluster_id),
                attempt=attempt + 1,
                attempts=SEARCH_RETRIES,
                delay=delay,
                error=f"{type(exc).__name__}: {exc}",
            )
            tqdm.write(
                f"STAC window failed for cluster {cluster_id} "
                f"(attempt {attempt + 1}/{SEARCH_RETRIES}): "
                f"{type(exc).__name__}: {exc}; retrying in {delay}s."
            )
            time.sleep(delay)
    return []


def _cluster_sample_day_union(cluster_stations):
    """Union of sample dates for a cluster, built once per cluster search."""
    arrays = [
        SAMPLE_DAYS.get(str(station_id))
        for station_id in cluster_stations["station_id"].astype(str).to_numpy()
    ]
    arrays = [a for a in arrays if a is not None and len(a)]
    if not arrays:
        return np.empty(0, dtype=np.int64)
    if len(arrays) == 1:
        return arrays[0]
    return np.unique(np.concatenate(arrays)).astype(np.int64, copy=False)


def _item_near_any_sample_day(item, cluster_days):
    """Fast temporal relevance test against the cluster-wide sorted date union."""
    if cluster_days is None or len(cluster_days) == 0:
        return False
    scene_day = item_day(item)
    k = int(np.searchsorted(cluster_days, scene_day, side="left"))
    nearest = np.inf
    if k < len(cluster_days):
        nearest = min(nearest, abs(int(cluster_days[k]) - scene_day))
    if k > 0:
        nearest = min(nearest, abs(int(cluster_days[k - 1]) - scene_day))
    return nearest <= MAX_NEAREST_DAYS


def _filter_cluster_items(items, cluster_row):
    """Keep relevant scenes using cached cluster dates and prebuilt station boxes."""
    cluster_id = int(cluster_row.cluster_id)
    cluster_stations = stations_cluster[
        stations_cluster["cluster_id"] == cluster_id
    ].reset_index(drop=True)
    if cluster_stations.empty:
        return []

    # This is a conservative filter: a date used by any station can admit a scene,
    # but the downstream per-station filter still enforces the exact sample window.
    cluster_days = _cluster_sample_day_union(cluster_stations)
    station_boxes = station_patch_boxes_deg(cluster_stations)
    station_geometries = shapely.box(
        station_boxes[:, 0],
        station_boxes[:, 1],
        station_boxes[:, 2],
        station_boxes[:, 3],
    )

    unique = {}
    for item in items:
        if not has_usable_polarizations(item):
            continue
        try:
            if not _item_near_any_sample_day(item, cluster_days):
                continue
            scene_geometry = _item_spatial_geometry(item)
            if scene_geometry is not None and not np.any(
                shapely.intersects(scene_geometry, station_geometries)
            ):
                continue
            unique[item.id] = item
        except (ValueError, TypeError, KeyError):
            continue
    return list(unique.values())


def _merged_sample_date_windows(cluster_stations, radius_days=MAX_NEAREST_DAYS):
    """Merge the union of [sample_day-radius, sample_day+radius] windows.

    The old strategy queried one continuous min_date..max_date interval for a
    cluster. If target dates were years apart, it downloaded metadata for all
    the dates in between. These merged windows include every date that can
    possibly contribute within the configured nearest-date tolerance, without
    querying those irrelevant gaps.
    """
    days = _cluster_sample_day_union(cluster_stations)
    if days is None or len(days) == 0:
        return []

    radius_days = int(radius_days)
    raw = sorted(
        (int(day) - radius_days, int(day) + radius_days)
        for day in np.asarray(days, dtype=np.int64)
    )
    merged = []
    for left, right in raw:
        if not merged or left > merged[-1][1] + 1:
            merged.append([left, right])
        else:
            merged[-1][1] = max(merged[-1][1], right)
    return [(int(left), int(right)) for left, right in merged]


def _query_cache_path(cluster_id, pair, start, end):
    """Stable cache key for one bounded STAC request."""
    start_day = pd.Timestamp(start).normalize().strftime("%Y%m%d")
    end_day = pd.Timestamp(end).normalize().strftime("%Y%m%d")
    pair_tag = "".join(str(x).upper() for x in pair)
    query_dir = STAC_CACHE_DIR / "query_pages"
    query_dir.mkdir(parents=True, exist_ok=True)
    return query_dir / (
        f"cluster_{int(cluster_id):05d}_{pair_tag}_{start_day}_{end_day}.json"
    )


def _cached_query_window_with_retry(bbox, start, end, cluster_id):
    """Resume a bounded window from cache; fall back to older paired-query caches."""
    all_path = _query_cache_path(cluster_id, ("ALL",), start, end)
    cached = load_stac_cache(all_path)
    if cached is not None:
        _emit_inventory_event(
            "query_cache_hit",
            cluster_id=int(cluster_id),
            start=str(pd.Timestamp(start).date()),
            end=str(pd.Timestamp(end).date()),
            items=len(cached),
            source="all-window cache",
        )
        return cached

    # Compatibility: if a previous run completed both polarization-specific
    # requests for exactly this window, combine them instead of downloading again.
    cached_vv = load_stac_cache(
        _query_cache_path(cluster_id, ("VV", "VH"), start, end)
    )
    cached_hh = load_stac_cache(
        _query_cache_path(cluster_id, ("HH", "HV"), start, end)
    )
    if cached_vv is not None and cached_hh is not None:
        by_id = {str(item.id): item for item in cached_vv}
        by_id.update({str(item.id): item for item in cached_hh})
        combined = list(by_id.values())
        save_stac_cache(all_path, combined)
        _emit_inventory_event(
            "query_cache_hit",
            cluster_id=int(cluster_id),
            start=str(pd.Timestamp(start).date()),
            end=str(pd.Timestamp(end).date()),
            items=len(combined),
            source="combined prior pair caches",
        )
        return combined

    items = _query_window_with_retry(bbox, start, end, cluster_id)
    save_stac_cache(all_path, items)
    return items


def search_cluster_full(cluster_row):
    """Search merged date windows; emit stages before doing potentially slow work."""
    cluster_id = int(cluster_row.cluster_id)
    _emit_inventory_event(
        "cluster_stage", cluster_id=cluster_id, stage="checking cluster cache"
    )
    try:
        new_cache = cluster_cache_path(cluster_id)
        cached = load_stac_cache(new_cache)
        if cached is not None:
            _emit_inventory_event(
                "cluster_stage", cluster_id=cluster_id,
                stage=f"cluster cache loaded ({len(cached):,} items)"
            )
            return cached

        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id, stage="preparing cluster stations"
        )
        cluster_stations = stations_cluster[
            stations_cluster["cluster_id"] == cluster_id
        ].reset_index(drop=True)
        if cluster_stations.empty:
            save_stac_cache(new_cache, [])
            return []

        bbox = [
            float(cluster_row.bbox_min_lon),
            float(cluster_row.bbox_min_lat),
            float(cluster_row.bbox_max_lon),
            float(cluster_row.bbox_max_lat),
        ]
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage="merging sample-date windows"
        )
        windows = _merged_sample_date_windows(cluster_stations)
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"date windows ready ({len(windows):,})"
        )
        if not windows:
            save_stac_cache(new_cache, [])
            return []

        def day_ts(day):
            return EPOCH + pd.Timedelta(days=int(day))

        found = {}
        for wi, (left_day, right_day) in enumerate(windows, start=1):
            _emit_inventory_event(
                "cluster_stage", cluster_id=cluster_id,
                stage=f"requesting date window {wi}/{len(windows)}"
            )
            # Inclusive target days are represented by an exclusive next-midnight end.
            window_items = _cached_query_window_with_retry(
                bbox,
                day_ts(left_day),
                day_ts(int(right_day) + 1),
                cluster_id,
            )
            for item in window_items:
                found[str(item.id)] = item

        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"filtering {len(found):,} candidate scenes"
        )
        items = _filter_cluster_items(list(found.values()), cluster_row)
        items = sorted(items, key=lambda item: (item_day(item), str(item.id)))
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"writing cache ({len(items):,} relevant scenes)"
        )
        save_stac_cache(new_cache, items)
        return items
    finally:
        _emit_inventory_event("cluster_end", cluster_id=cluster_id)


def open_scene_inventory_db():
    conn = sqlite3.connect(
        SCENE_INDEX_DB_PATH,
        timeout=120,
    )
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA temp_store=MEMORY")
    conn.execute("PRAGMA busy_timeout=120000")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS scenes (
            scene_id TEXT PRIMARY KEY,
            day INTEGER NOT NULL,
            item_json TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS scene_clusters (
            scene_id TEXT NOT NULL,
            cluster_id INTEGER NOT NULL,
            PRIMARY KEY(scene_id, cluster_id)
        );
        CREATE TABLE IF NOT EXISTS cluster_searches (
            cluster_id INTEGER PRIMARY KEY,
            completed INTEGER NOT NULL,
            result_count INTEGER NOT NULL,
            completed_utc TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_scenes_day ON scenes(day);
        CREATE INDEX IF NOT EXISTS idx_sc_cluster ON scene_clusters(cluster_id, scene_id);
        CREATE INDEX IF NOT EXISTS idx_sc_scene ON scene_clusters(scene_id, cluster_id);
        """
    )
    conn.commit()
    return conn


def _commit_cluster_items(conn, cluster_id, items):
    """Bulk-index a cluster and return the number of genuinely new scene rows."""
    if not items:
        with conn:
            conn.execute(
                """
                INSERT OR REPLACE INTO cluster_searches
                    (cluster_id, completed, result_count, completed_utc)
                VALUES (?, 1, 0, ?)
                """,
                (int(cluster_id), pd.Timestamp.now(tz="UTC").isoformat()),
            )
        return 0

    # Get existing IDs in safe-sized chunks. Avoid JSON serialization for the many
    # scene/cluster overlaps that have already been indexed by earlier clusters.
    unique_items = {str(item.id): item for item in items}
    ids = list(unique_items)
    existing_ids = set()
    for begin in range(0, len(ids), 800):
        batch = ids[begin:begin + 800]
        placeholders = ",".join(["?"] * len(batch))
        existing_ids.update(
            row[0]
            for row in conn.execute(
                f"SELECT scene_id FROM scenes WHERE scene_id IN ({placeholders})",
                batch,
            )
        )

    new_scene_rows = []
    for scene_id, item in unique_items.items():
        if scene_id in existing_ids:
            continue
        new_scene_rows.append((
            scene_id,
            item_day(item),
            json.dumps(item.to_dict(), separators=(",", ":")),
        ))

    association_rows = [
        (scene_id, int(cluster_id))
        for scene_id in unique_items
    ]

    with conn:
        if new_scene_rows:
            conn.executemany(
                "INSERT OR IGNORE INTO scenes(scene_id, day, item_json) VALUES (?, ?, ?)",
                new_scene_rows,
            )
        conn.executemany(
            "INSERT OR IGNORE INTO scene_clusters(scene_id, cluster_id) VALUES (?, ?)",
            association_rows,
        )
        conn.execute(
            """
            INSERT OR REPLACE INTO cluster_searches
                (cluster_id, completed, result_count, completed_utc)
            VALUES (?, 1, ?, ?)
            """,
            (
                int(cluster_id),
                len(items),
                pd.Timestamp.now(tz="UTC").isoformat(),
            ),
        )

    return len(new_scene_rows)


def build_global_scene_index(force=False):
    """Build the resumable date-window inventory with live request and scene heartbeats."""
    global _STAC_PROGRESS_EVENTS
    _STAC_PROGRESS_EVENTS = queue.Queue()

    cluster_rows = list(clusters.itertuples(index=False))
    all_ids = {int(row.cluster_id) for row in cluster_rows}
    conn = open_scene_inventory_db()

    if force:
        with conn:
            conn.execute("DELETE FROM scene_clusters")
            conn.execute("DELETE FROM scenes")
            conn.execute("DELETE FROM cluster_searches")

    completed = {
        int(row[0])
        for row in conn.execute(
            "SELECT cluster_id FROM cluster_searches WHERE completed = 1"
        )
    }
    pending = [row for row in cluster_rows if int(row.cluster_id) not in completed]
    existing_scene_count = int(
        conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
    )

    print("Clusters total:", f"{len(cluster_rows):,}")
    print("Clusters already complete:", f"{len(completed):,}")
    print("Clusters remaining:", f"{len(pending):,}")
    print("Unique scenes already indexed:", f"{existing_scene_count:,}")
    print("STAC search workers:", SEARCH_WORKERS)
    print("Scene workers:", SCENE_WORKERS)
    print("STAC request timeout:", STAC_REQUEST_TIMEOUT_SECONDS, "seconds")
    print("Window radius:", MAX_NEAREST_DAYS, "days around each sample date")

    if not pending:
        conn.close()
        _STAC_PROGRESS_EVENTS = None
        print("Inventory is already complete; no cluster searches to repeat.")
        return SCENE_INDEX_DB_PATH

    streaming_ready = STREAM_SCENES_DURING_INVENTORY and all(
        name in globals()
        for name in ("_scene_shard_path", "run_scene_record", "GLOBAL_STATION_TREE")
    )
    scene_executor = (
        ThreadPoolExecutor(max_workers=SCENE_WORKERS)
        if streaming_ready else None
    )
    max_scene_inflight = max(SCENE_WORKERS, min(SCENE_WORKERS * 2, 6))
    scene_futures = {}
    scene_started_at = {}
    scheduled_scene_ids = set()
    accounted_scene_ids = set()
    queued_scene_ids = set()
    scene_queue = deque()
    scene_failures = []
    scene_successes = 0
    running_scene_count = existing_scene_count

    # Store only IDs in the seed queue, not thousands of decoded item JSON objects.
    seed_scene_ids = deque(
        row[0] for row in conn.execute("SELECT scene_id FROM scenes ORDER BY rowid")
    ) if streaming_ready else deque()
    # Prevent scenes already in the initial backlog from also being queued again
    # when newly completed clusters report the same scene IDs.
    queued_scene_ids = set(seed_scene_ids)

    if streaming_ready:
        print(
            "Streaming extraction enabled: at most",
            max_scene_inflight,
            "scene jobs queued/in flight; one global spatial station index per scene.",
        )
    else:
        print("Scene extraction is decoupled from inventory; all missing shards are handled in stage 2.")

    active_queries = {}
    active_cluster_stages = {}
    inventory_started = time.monotonic()
    query_stats = {
        "started": 0,
        "finished": 0,
        "cache_hits": 0,
        "raw_items": 0,
        "failed": 0,
        "done": 0,
        "last_retry": "none",
        "last_event": time.monotonic(),
    }

    def drain_query_events(query_bar):
        while True:
            try:
                event = _STAC_PROGRESS_EVENTS.get_nowait()
            except queue.Empty:
                break

            kind = event.get("kind")
            query_stats["last_event"] = float(event.get("at", time.monotonic()))

            if kind == "cluster_stage":
                cid = int(event.get("cluster_id", -1))
                previous = active_cluster_stages.get(cid, {})
                active_cluster_stages[cid] = {
                    "stage": str(event.get("stage", "working")),
                    "started_at": previous.get(
                        "started_at", float(event.get("at", time.monotonic()))
                    ),
                }

            elif kind == "cluster_end":
                active_cluster_stages.pop(int(event.get("cluster_id", -1)), None)

            elif kind == "query_start":
                query_stats["started"] += 1
                active_queries[event["query_id"]] = {
                    **event,
                    "started_at": float(event.get("at", time.monotonic())),
                    "items_seen": 0,
                }
                query_bar.total = query_stats["started"] + query_stats["cache_hits"]
                query_bar.refresh()

            elif kind == "query_items":
                active = active_queries.get(event.get("query_id"))
                if active is not None:
                    active["items_seen"] = int(event.get("items_seen", 0))

            elif kind == "query_end":
                active_queries.pop(event.get("query_id"), None)
                query_stats["finished"] += 1
                query_stats["done"] += 1
                query_stats["raw_items"] += int(event.get("items", 0))
                if not event.get("success", False):
                    query_stats["failed"] += 1
                query_bar.update(1)

            elif kind == "query_cache_hit":
                query_stats["cache_hits"] += 1
                query_stats["done"] += 1
                query_stats["raw_items"] += int(event.get("items", 0))
                query_bar.total = query_stats["started"] + query_stats["cache_hits"]
                query_bar.update(1)

            elif kind == "query_retry":
                query_stats["last_retry"] = (
                    f"C{event.get('cluster_id')} "
                    f"attempt {event.get('attempt')}/{event.get('attempts')} "
                    f"in {event.get('delay')}s"
                )

    def update_live_bars(cluster_bar, query_bar, scene_bar, cluster_futures,
                         cluster_started_at, scene_started_at, scene_queue,
                         seed_scene_ids, running_scene_count, scene_successes):
        drain_query_events(query_bar)
        now = time.monotonic()

        active = sorted(
            active_queries.values(),
            key=lambda x: x.get("started_at", now),
        )
        if active:
            oldest_q = active[0]
            q_current = (
                f"C{oldest_q.get('cluster_id')} "
                f"{oldest_q.get('start')}..{oldest_q.get('end')}"
            )
            q_age = f"{int(now - oldest_q.get('started_at', now))}s"
            q_items = int(oldest_q.get("items_seen", 0))
        else:
            q_current, q_age, q_items = "no request active", "-", 0

        query_bar.set_postfix(
            active=len(active),
            current=q_current,
            age=q_age,
            items=q_items,
            raw=f"{query_stats['raw_items']:,}",
            cache=f"{query_stats['cache_hits']:,}",
            retries=query_stats["last_retry"],
            alive=f"{int(now - query_stats['last_event'])}s",
            refresh=True,
        )

        if cluster_futures:
            oldest_cluster_future = min(
                cluster_futures,
                key=lambda future: cluster_started_at.get(future, now),
            )
            oldest_row = cluster_futures[oldest_cluster_future]
            cluster_label = (
                f"C{int(oldest_row.cluster_id)} "
                f"{int(now - cluster_started_at.get(oldest_cluster_future, now))}s"
            )
        else:
            cluster_label = "none"

        stage_rows = sorted(
            active_cluster_stages.items(),
            key=lambda pair: pair[1].get("started_at", now),
        )
        stage_labels = [
            f"C{cid}:{entry.get('stage', 'working')}:{int(now - entry.get('started_at', now))}s"
            for cid, entry in stage_rows[:2]
        ]
        cluster_bar.set_postfix(
            checkpoints=f"{len(completed):,}/{len(all_ids):,}",
            active=len(cluster_futures),
            oldest=cluster_label,
            phase=" | ".join(stage_labels) or "no active stage",
            unique=f"{running_scene_count:,}",
            req=f"{query_stats['done']:,}/{query_stats['started'] + query_stats['cache_hits']:,}",
            refresh=True,
        )

        if scene_bar is not None:
            active_scene_rows = sorted(
                scene_futures.items(),
                key=lambda pair: scene_started_at.get(pair[0], now),
            )
            active_labels = []
            oldest_age = "-"
            for future, scene_id in active_scene_rows[:2]:
                age = int(now - scene_started_at.get(future, now))
                active_labels.append(f"{str(scene_id)[:16]}:{age}s")
            if active_scene_rows:
                oldest_age = f"{int(now - scene_started_at.get(active_scene_rows[0][0], now))}s"
            scene_bar.set_postfix(
                done=f"{scene_bar.n:,}/{int(scene_bar.total or 0):,}",
                discovered=f"{running_scene_count:,}",
                queued=len(scene_queue) + len(seed_scene_ids),
                active=len(scene_futures),
                current=", ".join(active_labels) or "none",
                oldest=oldest_age,
                elapsed=f"{int(now - inventory_started)}s",
                heartbeat="alive",
                refresh=True,
            )

    scene_bar = None
    cluster_bar = None
    query_bar = None
    pending_iter = iter(pending)
    cluster_futures = {}
    cluster_started_at = {}
    failures = []

    def submit_cluster(executor, row):
        future = executor.submit(search_cluster_full, row)
        cluster_futures[future] = row
        cluster_started_at[future] = time.monotonic()

    def account_scene(scene_id, bar):
        if scene_id in accounted_scene_ids:
            return
        accounted_scene_ids.add(scene_id)
        if bar is not None:
            bar.update(1)

    def pump_scene_jobs():
        if not streaming_ready:
            return
        checks_this_pass = 0
        max_checks_per_pass = 32
        while (
            len(scene_futures) < max_scene_inflight
            and checks_this_pass < max_checks_per_pass
        ):
            scene_id = None
            if scene_queue:
                scene_id = scene_queue.popleft()
                queued_scene_ids.discard(scene_id)
            elif seed_scene_ids:
                scene_id = seed_scene_ids.popleft()
                queued_scene_ids.discard(str(scene_id))
            else:
                break

            checks_this_pass += 1
            scene_id = str(scene_id)
            if not scene_id or scene_id in scheduled_scene_ids:
                continue

            if _scene_shard_path(scene_id).exists():
                scheduled_scene_ids.add(scene_id)
                account_scene(scene_id, scene_bar)
                continue

            row = conn.execute(
                "SELECT item_json FROM scenes WHERE scene_id = ?",
                (scene_id,),
            ).fetchone()
            if row is None:
                continue

            record = {
                "item": json.loads(row[0]),
                "cluster_ids": [],
                "global_scope": True,
            }
            future = scene_executor.submit(run_scene_record, record)
            scene_futures[future] = scene_id
            scene_started_at[future] = time.monotonic()
            scheduled_scene_ids.add(scene_id)

    def handle_scene_completions(done_futures):
        nonlocal scene_successes
        if not streaming_ready:
            return
        for future in list(done_futures):
            if future not in scene_futures:
                continue
            scene_id = scene_futures.pop(future)
            scene_started_at.pop(future, None)
            try:
                result = future.result()
                scene_successes += 1
                account_scene(scene_id, scene_bar)
                if scene_bar is not None:
                    scene_bar.set_postfix(
                        status=result.get("status", "ok"),
                        last=str(scene_id)[:18],
                        discovered=f"{running_scene_count:,}",
                        refresh=True,
                    )
            except KeyboardInterrupt:
                raise
            except Exception as exc:
                scene_failures.append((scene_id, f"{type(exc).__name__}: {exc}"))
                scheduled_scene_ids.discard(scene_id)
                tqdm.write(
                    f"Streaming scene {scene_id} failed (will retry in final extraction phase): "
                    f"{type(exc).__name__}: {exc}"
                )

    try:
        with ThreadPoolExecutor(max_workers=SEARCH_WORKERS) as search_executor:
            cluster_bar = tqdm(
                total=len(pending),
                desc="Inventory clusters checkpointed",
                unit="cluster",
                position=0,
                leave=True,
                dynamic_ncols=True,
                mininterval=0.5,
            )
            query_bar = tqdm(
                total=0,
                desc="STAC date-window requests",
                unit="req",
                position=1,
                leave=True,
                dynamic_ncols=True,
                mininterval=0.5,
            )
            if streaming_ready:
                scene_bar = tqdm(
                    total=existing_scene_count,
                    desc="Scene shards completed",
                    unit="scene",
                    position=2,
                    leave=True,
                    dynamic_ncols=True,
                    mininterval=0.5,
                )

            for _ in range(min(SEARCH_WORKERS, len(pending))):
                row = next(pending_iter, None)
                if row is not None:
                    submit_cluster(search_executor, row)

            while cluster_futures:
                update_live_bars(
                    cluster_bar, query_bar, scene_bar,
                    cluster_futures, cluster_started_at,
                    scene_started_at, scene_queue, seed_scene_ids,
                    running_scene_count, scene_successes,
                )
                watch = set(cluster_futures) | set(scene_futures)
                done_futures, _ = wait(
                    watch,
                    timeout=1.0,
                    return_when=FIRST_COMPLETED,
                )
                handle_scene_completions(done_futures)

                for future in list(done_futures):
                    if future not in cluster_futures:
                        continue
                    row = cluster_futures.pop(future)
                    cluster_started_at.pop(future, None)
                    cluster_id = int(row.cluster_id)
                    cluster_item_count = 0
                    try:
                        items = future.result()
                        cluster_item_count = len(items)
                        newly_indexed = _commit_cluster_items(conn, cluster_id, items)
                        running_scene_count += newly_indexed
                        completed.add(cluster_id)

                        if scene_bar is not None and newly_indexed:
                            scene_bar.total = int(scene_bar.total or 0) + newly_indexed
                            scene_bar.refresh()

                        if streaming_ready:
                            # Queue only scene IDs, not decoded STAC JSON. The JSON is
                            # fetched lazily only when a worker slot is ready.
                            for item in items:
                                scene_id = str(item.id)
                                if _scene_shard_path(scene_id).exists():
                                    scheduled_scene_ids.add(scene_id)
                                    account_scene(scene_id, scene_bar)
                                    continue
                                if (
                                    scene_id not in scheduled_scene_ids
                                    and scene_id not in queued_scene_ids
                                ):
                                    scene_queue.append(scene_id)
                                    queued_scene_ids.add(scene_id)
                    except KeyboardInterrupt:
                        raise
                    except Exception as exc:
                        failures.append(cluster_id)
                        tqdm.write(
                            f"Cluster {cluster_id} failed: {type(exc).__name__}: {exc}"
                        )

                    cluster_bar.update(1)
                    cluster_bar.set_postfix(
                        checkpointed=f"{len(completed):,}/{len(all_ids):,}",
                        unique_scenes=f"{running_scene_count:,}",
                        returned=cluster_item_count,
                        stream_ok=scene_successes,
                        refresh=True,
                    )

                    next_row = next(pending_iter, None)
                    if next_row is not None:
                        submit_cluster(search_executor, next_row)

                pump_scene_jobs()
                update_live_bars(
                    cluster_bar, query_bar, scene_bar,
                    cluster_futures, cluster_started_at,
                    scene_started_at, scene_queue, seed_scene_ids,
                    running_scene_count, scene_successes,
                )

            # Inventory searches are done. Drain all scene IDs discovered during
            # this run plus the database backlog from any interrupted earlier run.
            if streaming_ready:
                pump_scene_jobs()
                while scene_futures or scene_queue or seed_scene_ids:
                    if scene_futures:
                        done_futures, _ = wait(
                            list(scene_futures),
                            timeout=1.0,
                            return_when=FIRST_COMPLETED,
                        )
                        handle_scene_completions(done_futures)
                    pump_scene_jobs()
                    update_live_bars(
                        cluster_bar, query_bar, scene_bar,
                        cluster_futures, cluster_started_at,
                        scene_started_at, scene_queue, seed_scene_ids,
                        running_scene_count, scene_successes,
                    )

            # Drain final query status events before closing the progress bars.
            drain_query_events(query_bar)

    finally:
        for bar in (cluster_bar, query_bar, scene_bar):
            if bar is not None:
                bar.close()
        if scene_executor is not None:
            scene_executor.shutdown(wait=True)
        conn.close()
        _STAC_PROGRESS_EVENTS = None

    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as stats_conn:
        unique_count = int(
            stats_conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
        )
    with sqlite3.connect(SCENE_INDEX_DB_PATH) as stats_conn:
        association_count = int(
            stats_conn.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0]
        )
        committed_count = int(
            stats_conn.execute(
                "SELECT COUNT(*) FROM cluster_searches WHERE completed=1"
            ).fetchone()[0]
        )

    print("Unique scenes:", f"{unique_count:,}")
    print("Scene-cluster associations:", f"{association_count:,}")
    print("Clusters committed:", f"{committed_count:,}/{len(all_ids):,}")
    print("Inventory database:", SCENE_INDEX_DB_PATH)
    if scene_failures:
        print(
            f"Streaming scene failures: {len(scene_failures)}. "
            "The final extraction stage will retry missing shards. "
            f"First failures: {scene_failures[:5]}"
        )
    if failures:
        raise RuntimeError(
            f"{len(failures)} cluster searches failed and remain resumable. "
            "Rerun the inventory cell after resolving the errors. "
            f"Failed cluster IDs (first 20): {[x for x in failures[:20]]}"
        )
    if completed != all_ids:
        raise RuntimeError(
            f"Inventory incomplete: {len(completed)} of {len(all_ids)} clusters committed."
        )
    return SCENE_INDEX_DB_PATH


def load_global_scene_index():
    """Compatibility alias; returns the SQLite index path, not a giant Python list."""
    return build_global_scene_index()


## Scene-centric production acquisition

The production unit is one unique Sentinel-1 RTC scene. Cluster queries are written incrementally to SQLite; repeated scenes are stored once and associated with every relevant cluster. The inventory uses merged intervals around actual sample dates, not one continuous min-to-max span, and caches each bounded interval atomically.

The inventory UI has three independent progress indicators: completed cluster checkpoints, individual STAC interval requests (including active request age, item counts, cache hits, and retries), and scene shards completed while inventory continues. A live scene heartbeat is shown while remote COG reads are in flight. A bounded queue keeps raster concurrency and memory use controlled. The STAC client uses a finite request timeout and retries.

For each sample, selection uses the closest valid acquisition within `MAX_NEAREST_DAYS` before or after the sample date; an equal-distance tie favors the earlier date. There is no temporal median.

In [33]:
# ============================================================
# SCENE SELECTION
# ============================================================

def item_day(item):
    dt = (
        item.datetime
        or item.properties.get("start_datetime")
        or item.properties.get("end_datetime")
    )
    if dt is None:
        raise ValueError(
            f"STAC item {item.id} has no acquisition datetime"
        )
    ts = pd.Timestamp(dt)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return int((ts.normalize() - EPOCH).days)


def available_polarizations(item):
    asset_names = {
        str(name).lower()
        for name in item.assets
    }
    return [pol for pol in REAL_BANDS if pol in asset_names]


def has_usable_polarizations(item):
    """Production extraction only uses VV/VH; HH/HV-only scenes are skipped."""
    return set(READ_POLARIZATIONS).issubset(set(available_polarizations(item)))


def has_required_polarizations(item):
    """VV/VH-only compatibility helper for diagnostics."""
    return {"vv", "vh"}.issubset(set(available_polarizations(item)))


def _item_spatial_geometry(item):
    """Return exact STAC geometry when present, otherwise its conservative bbox."""
    if item.geometry is not None:
        return shapely.geometry.shape(item.geometry)

    bbox = getattr(item, "bbox", None)
    if bbox is not None and len(bbox) >= 4:
        return shapely.box(
            float(bbox[0]), float(bbox[1]),
            float(bbox[2]), float(bbox[3]),
        )
    return None


def station_patch_intersects_item(item, station_df):
    item_geometry = _item_spatial_geometry(item)
    if item_geometry is None:
        # No usable footprint: retain the safe, conservative previous behavior.
        return np.ones(len(station_df), dtype=bool)

    boxes = station_patch_boxes_deg(station_df)
    geometries = shapely.box(
        boxes[:, 0],
        boxes[:, 1],
        boxes[:, 2],
        boxes[:, 3],
    )
    return np.asarray(
        shapely.intersects(item_geometry, geometries),
        dtype=bool,
    )


def _nearest_day_distance(days, target_day):
    """Minimum absolute day distance in a sorted int array."""
    if days is None or len(days) == 0:
        return np.inf
    k = int(np.searchsorted(days, target_day, side="left"))
    distance = np.inf
    if k < len(days):
        distance = min(distance, abs(int(days[k]) - int(target_day)))
    if k > 0:
        distance = min(distance, abs(int(days[k - 1]) - int(target_day)))
    return distance


def stations_needed_for_scene(item, station_df, sample_days):
    """
    Return stations whose spatial patch intersects the scene and which have
    at least one sample date within +/- MAX_NEAREST_DAYS of the acquisition.
    """
    if not has_usable_polarizations(item):
        return station_df.iloc[0:0].copy()

    spatial = station_patch_intersects_item(item, station_df)
    scene_day = item_day(item)
    station_ids = station_df["station_id"].astype(str).to_numpy()

    needed = np.zeros(len(station_df), dtype=bool)
    for j, station_id in enumerate(station_ids):
        if not spatial[j]:
            continue
        days = sample_days.get(station_id)
        if _nearest_day_distance(days, scene_day) <= MAX_NEAREST_DAYS:
            needed[j] = True

    return station_df.loc[needed].reset_index(drop=True)


## Search and scene selection

The full cluster query is filtered by the actual station-patch envelopes and sample dates. For production raster extraction, an RTC scene is retained only if at least one station can plausibly use it within the configured nearest-date tolerance and it provides the required VV/VH pair. HH/HV assets are not downloaded.

The final compiler checks actual raster validity and selects the closest usable scene. Exact temporal ties prefer the earlier acquisition.


In [34]:
# ============================================================
# NATIVE 10m WINDOWS
# ============================================================

_tls = threading.local()


def transformer_to(
    crs_string,
):
    cache = getattr(
        _tls,
        "transformers",
        None,
    )

    if cache is None:
        cache = {}
        _tls.transformers = cache

    if crs_string not in cache:

        cache[
            crs_string
        ] = Transformer.from_crs(
            "EPSG:4326",
            crs_string,
            always_xy=True,
        )

    return cache[
        crs_string
    ]


def station_patch_windows(
    transform,
    crs_string,
    width,
    height,
    station_df,
):
    """
    Exact 1300x1300 native-pixel windows
    centered on each station.
    """

    transformer = (
        transformer_to(
            crs_string
        )
    )

    x, y = transformer.transform(
        station_df[
            "longitude"
        ].to_numpy(float),

        station_df[
            "latitude"
        ].to_numpy(float),
    )

    inv = ~transform

    cols = (
        inv.a * x
        + inv.b * y
        + inv.c
    )

    rows = (
        inv.d * x
        + inv.e * y
        + inv.f
    )

    c0 = (
        np.floor(cols)
        .astype(np.int64)
        - HALF_PATCH_PX
    )

    r0 = (
        np.floor(rows)
        .astype(np.int64)
        - HALF_PATCH_PX
    )

    c1 = (
        c0
        + SOURCE_PATCH_PX
    )

    r1 = (
        r0
        + SOURCE_PATCH_PX
    )

    pixels = np.column_stack(
        [
            c0,
            r0,
            c1,
            r1,
        ]
    )

    valid = (
        (c1 > 0)
        &
        (r1 > 0)
        &
        (c0 < width)
        &
        (r0 < height)
    )

    return (
        pixels,
        valid,
    )

In [35]:
# ============================================================
# COG TILE PLANNER -- MEMORY-SAFE
# ============================================================

def scene_layout(src):
    block_h, block_w = src.block_shapes[0]
    if not src.profile.get("tiled", False):
        raise RuntimeError("Sentinel-1 RTC asset is unexpectedly not tiled.")
    return {
        "width": src.width,
        "height": src.height,
        "transform": src.transform,
        "crs": src.crs.to_string(),
        "block_w": int(block_w),
        "block_h": int(block_h),
        "dtype": src.dtypes[0],
        "nodata": src.nodata,
        "overviews": src.overviews(1),
        "key": (src.width, src.height, tuple(src.transform)[:6]),
    }


def merge_tile_boxes(tile_boxes, slack=MERGE_SLACK):
    """
    Greedy O(n log n) merge of tile-aligned boxes.

    The previous implementation built several n x n matrices. With many
    clusters contributing to one scene that can exhaust notebook RAM and
    kill the kernel. This version sorts boxes spatially and grows one local
    merge group at a time, so memory stays O(n).
    """
    boxes = np.asarray(tile_boxes, dtype=np.int64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes.copy(), []
    if len(boxes) == 1:
        return boxes.copy(), [[0]]

    order = np.lexsort((boxes[:, 1], boxes[:, 0], boxes[:, 3], boxes[:, 2]))
    sorted_boxes = boxes[order]

    merged = []
    groups = []
    cur = None
    cur_group = []
    cur_area_sum = 0

    for local_sorted_index, box in enumerate(sorted_boxes):
        x0, y0, x1, y1 = map(int, box)
        area = max(0, x1 - x0) * max(0, y1 - y0)

        if cur is None:
            cur = [x0, y0, x1, y1]
            cur_group = [int(order[local_sorted_index])]
            cur_area_sum = area
            continue

        ux0 = min(cur[0], x0)
        uy0 = min(cur[1], y0)
        ux1 = max(cur[2], x1)
        uy1 = max(cur[3], y1)
        union_area = max(0, ux1 - ux0) * max(0, uy1 - uy0)

        if union_area <= float(slack) * float(cur_area_sum + area):
            cur = [ux0, uy0, ux1, uy1]
            cur_group.append(int(order[local_sorted_index]))
            cur_area_sum += area
        else:
            merged.append(tuple(cur))
            groups.append(cur_group)
            cur = [x0, y0, x1, y1]
            cur_group = [int(order[local_sorted_index])]
            cur_area_sum = area

    merged.append(tuple(cur))
    groups.append(cur_group)

    return np.asarray(merged, dtype=np.int64), groups


def plan_scene_reads(layout, station_df):
    """Convert exact station patches to clipped, tile-aligned remote reads."""
    pixels, valid = station_patch_windows(
        layout["transform"],
        layout["crs"],
        layout["width"],
        layout["height"],
        station_df,
    )

    valid_indices = np.flatnonzero(valid)
    if not len(valid_indices):
        return {"layout": layout, "reads": []}

    p = pixels[valid_indices]
    bw = layout["block_w"]
    bh = layout["block_h"]

    tx0 = np.floor_divide(p[:, 0], bw)
    ty0 = np.floor_divide(p[:, 1], bh)
    tx1 = np.ceil(p[:, 2] / bw).astype(np.int64)
    ty1 = np.ceil(p[:, 3] / bh).astype(np.int64)

    max_tx = math.ceil(layout["width"] / bw)
    max_ty = math.ceil(layout["height"] / bh)

    tx0 = np.clip(tx0, 0, max_tx)
    ty0 = np.clip(ty0, 0, max_ty)
    tx1 = np.clip(tx1, 0, max_tx)
    ty1 = np.clip(ty1, 0, max_ty)

    tile_boxes = np.column_stack([tx0, ty0, tx1, ty1]).astype(np.int64)
    merged, groups = merge_tile_boxes(tile_boxes)

    station_ids = station_df["station_id"].to_numpy()
    reads = []

    for tile_box, group in zip(merged, groups):
        bx0, by0, bx1, by1 = map(int, tile_box)
        c0 = bx0 * bw
        r0 = by0 * bh
        c1 = min(bx1 * bw, layout["width"])
        r1 = min(by1 * bh, layout["height"])

        members = []
        for local_index in group:
            original_index = valid_indices[local_index]
            members.append(
                (
                    station_ids[original_index],
                    tuple(int(v) for v in pixels[original_index]),
                )
            )

        reads.append(
            {
                "window": Window(c0, r0, c1 - c0, r1 - r0),
                "members": members,
            }
        )

    return {"layout": layout, "reads": reads}


## Remote COG scene reading

Production defaults to direct 13 x 13 reads using COG overviews with average resampling. This is substantially faster in the representative benchmark, but it is approximate relative to averaging all native 10 m pixels. Change `PRODUCTION_READ_STRATEGY` in the configuration cell to `"exact"` to restore native 10 m reads and exact 100 x 100 pixel means.

The exact and overview strategies write to separate scene-shard and final-output folders. Do not manually copy shards across those folders.


In [36]:
# ============================================================
# REMOTE COG READ + 10m -> 1km AGGREGATION
# ============================================================


def _normalize_raster_values(data, nodata):
    """Convert nodata/non-finite values to NaN in a float32 array."""
    data = np.asarray(data, dtype=np.float32).copy()
    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if not np.isnan(nodata_value):
                data[data == np.float32(nodata_value)] = np.nan
        except (TypeError, ValueError):
            pass
    data[~np.isfinite(data)] = np.nan
    return data


def read_block(src, window):
    """Read a native-resolution block for the exact aggregation strategy."""
    data = src.read(
        1,
        window=window,
        out_dtype="float32",
    )
    return _normalize_raster_values(data, src.nodata)


def extract_member_patch(block, read_window, requested_pixels):
    """Extract an exact native 10 m station patch from a merged read."""
    c0, r0, c1, r1 = map(int, requested_pixels)
    patch = np.full((SOURCE_PATCH_PX, SOURCE_PATCH_PX), np.nan, dtype=np.float32)

    read_c0 = max(c0, int(read_window.col_off))
    read_r0 = max(r0, int(read_window.row_off))
    read_c1 = min(c1, int(read_window.col_off + read_window.width))
    read_r1 = min(r1, int(read_window.row_off + read_window.height))
    if read_c1 <= read_c0 or read_r1 <= read_r0:
        return patch

    src_c0 = read_c0 - int(read_window.col_off)
    src_r0 = read_r0 - int(read_window.row_off)
    src_c1 = read_c1 - int(read_window.col_off)
    src_r1 = read_r1 - int(read_window.row_off)
    dst_c0 = read_c0 - c0
    dst_r0 = read_r0 - r0
    dst_c1 = dst_c0 + (read_c1 - read_c0)
    dst_r1 = dst_r0 + (read_r1 - read_r0)
    patch[dst_r0:dst_r1, dst_c0:dst_c1] = block[src_r0:src_r1, src_c0:src_c1]
    return patch


def pool_10m_to_1km(patch):
    """1300x1300 native pixels -> 13x13 means over 100x100 pixel cells."""
    x = patch.reshape(
        PATCH_SIZE,
        SOURCE_PIXELS_PER_MODEL_CELL,
        PATCH_SIZE,
        SOURCE_PIXELS_PER_MODEL_CELL,
    )
    valid = np.isfinite(x)
    count = valid.sum(axis=(1, 3))
    total = np.nansum(x, axis=(1, 3), dtype=np.float64)
    output = np.full((PATCH_SIZE, PATCH_SIZE), np.nan, dtype=np.float32)
    np.divide(total, count, out=output, where=(count > 0))
    return output


def read_overview_patch(src, requested_pixels):
    """Read one station window directly to 13x13 using COG overviews.

    The 1300x1300 source window and 13x13 output preserve the station-grid
    alignment. Boundless reads pad areas beyond the raster footprint with NaN.
    This strategy is an approximation because the stored COG overviews were
    precomputed and need not equal a fresh native-pixel 100x100 mean.
    """
    if not src.overviews(1):
        raise RuntimeError(
            "PRODUCTION_READ_STRATEGY='overview' but this COG asset has no "
            f"overviews: {src.name}"
        )

    c0, r0, c1, r1 = map(int, requested_pixels)
    window = Window(c0, r0, c1 - c0, r1 - r0)
    data = src.read(
        1,
        window=window,
        out_shape=(PATCH_SIZE, PATCH_SIZE),
        out_dtype="float32",
        resampling=Resampling.average,
        boundless=True,
        fill_value=np.nan,
    )
    return _normalize_raster_values(data, src.nodata)


def read_remote_scene_band(item, asset_key, station_df, plan=None):
    """Read one RTC polarization using the configured exact/overview strategy."""
    for attempt in range(READ_RETRIES):
        try:
            href = rtc_token.sign(
                _clean_href(item.assets[asset_key].href),
                force=(attempt > 0),
            )
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(href) as src:
                    layout = scene_layout(src)
                    if plan is None or plan["layout"]["key"] != layout["key"]:
                        plan = plan_scene_reads(layout, station_df)

                    station_patches = {}

                    if PRODUCTION_READ_STRATEGY == "overview":
                        # Only one 13x13 overview-backed result is requested per
                        # station. This avoids downloading the full native 1300x1300
                        # source patch and keeps the output grid explicitly aligned.
                        for read in plan["reads"]:
                            for station_id, requested_pixels in read["members"]:
                                station_patches[str(station_id)] = read_overview_patch(
                                    src,
                                    requested_pixels,
                                )
                    else:
                        # Exact path: preserve the existing merged tile reads and
                        # native 10 m -> 1 km aggregation.
                        for read in plan["reads"]:
                            block = read_block(src, read["window"])
                            for station_id, requested_pixels in read["members"]:
                                native_patch = extract_member_patch(
                                    block,
                                    read["window"],
                                    requested_pixels,
                                )
                                station_patches[str(station_id)] = pool_10m_to_1km(
                                    native_patch
                                )

                    return station_patches, plan

        except (
            requests.RequestException,
            rasterio.errors.RasterioIOError,
            OSError,
            TimeoutError,
            ConnectionError,
            RuntimeError,
        ) as exc:
            if attempt >= READ_RETRIES - 1:
                raise
            delay = min(15, 2 ** attempt)
            tqdm.write(
                f"Remote read failed for {item.id}/{asset_key} "
                f"(attempt {attempt + 1}/{READ_RETRIES}): "
                f"{type(exc).__name__}: {exc}. Retrying in {delay}s."
            )
            time.sleep(delay)

    raise RuntimeError(f"Remote read failed for {item.id}/{asset_key}.")


In [37]:
# ============================================================
# SCENE-CENTRIC REMOTE COG WORKER
# ============================================================

def _scene_station_subset(scene_cluster_ids):
    """Return unique station rows linked to this globally unique scene."""
    subset = stations_cluster[
        stations_cluster["cluster_id"].isin(scene_cluster_ids)
    ][
        ["station_id", "latitude", "longitude", "cluster_id"]
    ].copy()
    if subset.empty:
        return subset
    return subset.drop_duplicates("station_id").reset_index(drop=True)


def _global_scene_station_candidates(item):
    """Return only station-patch envelopes intersecting this scene's footprint."""
    geometry = _item_spatial_geometry(item)
    if geometry is None:
        # Conservative fallback only for malformed STAC items with no bbox.
        return GLOBAL_STATION_FRAME.copy()
    try:
        indices = np.asarray(
            GLOBAL_STATION_TREE.query(geometry, predicate="intersects")
        )
        if not np.issubdtype(indices.dtype, np.integer):
            # Compatibility fallback for older STRtree APIs.
            indices = np.flatnonzero(
                np.asarray(
                    shapely.intersects(geometry, GLOBAL_STATION_PATCH_GEOMETRIES),
                    dtype=bool,
                )
            )
    except (TypeError, AttributeError):
        indices = np.flatnonzero(
            np.asarray(
                shapely.intersects(geometry, GLOBAL_STATION_PATCH_GEOMETRIES),
                dtype=bool,
            )
        )
    if len(indices) == 0:
        return GLOBAL_STATION_FRAME.iloc[0:0].copy()
    return GLOBAL_STATION_FRAME.iloc[np.unique(indices)].reset_index(drop=True)


def process_scene_remote(item, scene_cluster_ids=None):
    """Read a scene once for every globally relevant station (or explicit clusters)."""
    if not has_usable_polarizations(item):
        return None

    if scene_cluster_ids is None:
        station_df = _global_scene_station_candidates(item)
    else:
        station_df = _scene_station_subset(scene_cluster_ids)
    if station_df.empty:
        return None

    needed_stations = stations_needed_for_scene(
        item,
        station_df,
        SAMPLE_DAYS,
    )
    if needed_stations.empty:
        return None

    station_ids = needed_stations["station_id"].astype(str).to_numpy()
    output = np.full(
        (
            len(station_ids),
            N_REAL_BANDS,
            PATCH_SIZE,
            PATCH_SIZE,
        ),
        np.nan,
        dtype=np.float32,
    )

    station_to_row = {
        station_id: i
        for i, station_id in enumerate(station_ids)
    }

    available = set(available_polarizations(item))
    bands_to_read = [
        (band_index, pol)
        for band_index, pol in enumerate(REAL_BANDS)
        if pol in READ_POLARIZATIONS and pol in available
    ]

    if PARALLEL_POLARIZATION_READS and len(bands_to_read) > 1:
        # VV and VH are the only remote assets read in this production mode.
        # Read them concurrently so a slow HTTP range request for one asset does not hold
        # up the independent asset read. Each worker opens its own Rasterio
        # dataset and uses the unchanged v9 window/resampling logic.
        band_workers = min(int(POLARIZATION_READ_WORKERS), len(bands_to_read))
        with ThreadPoolExecutor(max_workers=band_workers) as band_executor:
            future_to_band = {
                band_executor.submit(
                    read_remote_scene_band,
                    item,
                    pol,
                    needed_stations,
                    None,
                ): (band_index, pol)
                for band_index, pol in bands_to_read
            }
            reference_layout_key = None
            for future in as_completed(future_to_band):
                band_index, pol = future_to_band[future]
                patches, band_plan = future.result()
                current_layout_key = band_plan["layout"]["key"]
                if reference_layout_key is None:
                    reference_layout_key = current_layout_key
                elif current_layout_key != reference_layout_key:
                    raise RuntimeError(
                        f"Polarization assets for scene {item.id} have mismatched grids; "
                        f"cannot combine {pol} with the other bands."
                    )
                for station_id, patch in patches.items():
                    output[station_to_row[str(station_id)], band_index] = patch
    else:
        # Sequential fallback reads only the configured VV/VH assets.
        plan = None
        for band_index, pol in bands_to_read:
            patches, plan = read_remote_scene_band(
                item,
                pol,
                needed_stations,
                plan=plan,
            )
            for station_id, patch in patches.items():
                output[station_to_row[str(station_id)], band_index] = patch

    return {
        "item_id": item.id,
        "day": item_day(item),
        "station_ids": station_ids,
        "data": output,
    }


def process_scene_with_retry(item, scene_cluster_ids, attempts=SCENE_RETRIES):
    """Retry transient remote raster failures; preserve completed scene shards."""
    for attempt in range(attempts):
        try:
            return process_scene_remote(
                item,
                scene_cluster_ids,
            )
        except KeyboardInterrupt:
            raise
        except (
            requests.RequestException,
            rasterio.errors.RasterioIOError,
            OSError,
            TimeoutError,
            ConnectionError,
        ) as exc:
            if attempt >= attempts - 1:
                raise
            delay = min(60, 2 ** (attempt + 1))
            tqdm.write(
                f"Scene {item.id} failed (attempt {attempt + 1}/{attempts}): "
                f"{type(exc).__name__}: {exc}. Retrying in {delay}s."
            )
            time.sleep(delay)
    return None


In [38]:
# ============================================================
# OPTIONAL REMOTE SMOKE TEST
# ============================================================

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    test_cluster = clusters.sort_values(
        "n_stations",
        ascending=False,
    ).iloc[0]
    TEST_CLUSTER_ID = int(test_cluster["cluster_id"])
    TEST_STATIONS = stations_cluster[
        stations_cluster["cluster_id"] == TEST_CLUSTER_ID
    ].reset_index(drop=True)

    TEST_ITEMS = search_cluster_full(test_cluster)
    candidate_scenes = []
    for item in TEST_ITEMS:
        needed = stations_needed_for_scene(
            item,
            TEST_STATIONS,
            SAMPLE_DAYS,
        )
        if not needed.empty:
            candidate_scenes.append((len(needed), item, needed))

    if not candidate_scenes:
        raise RuntimeError(
            "No useful Sentinel-1 RTC scene in the test cluster can contribute "
            "to a station sample within the nearest-date tolerance."
        )

    _, TEST_ITEM, TEST_NEEDED_STATIONS = max(
        candidate_scenes,
        key=lambda x: x[0],
    )
    TEST_NEEDED_STATIONS = TEST_NEEDED_STATIONS.head(5).reset_index(drop=True)

    print("Test cluster:", TEST_CLUSTER_ID)
    print("Scene:", TEST_ITEM.id)
    print("Stations tested:", len(TEST_NEEDED_STATIONS))
    print("Available polarizations:", available_polarizations(TEST_ITEM))

    result = process_scene_remote(
        TEST_ITEM,
        [TEST_CLUSTER_ID],
    )
    if result is None:
        raise RuntimeError("Smoke test scene returned no station data.")

    print("Smoke-test output shape:", result["data"].shape)
    print("Expected final dimensions:", (len(result["station_ids"]), N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE))
    print("Smoke test complete.")
else:
    print("Remote smoke test disabled; proceed to the global inventory cell when ready.")


Remote smoke test disabled; proceed to the global inventory cell when ready.


In [39]:
# ============================================================
# OPTIONAL REMOTE RANGE DIAGNOSTIC
# ============================================================

if "TEST_ITEM" not in globals():
    print("Skipping range diagnostic: TEST_ITEM is not defined. Enable the smoke test first if needed.")
else:
    available = [p for p in available_polarizations(TEST_ITEM) if p in READ_POLARIZATIONS]
    if not available:
        raise RuntimeError("TEST_ITEM has no required VV/VH polarization assets.")

    test_pol = available[0]
    test_href = rtc_token.sign(
        _clean_href(TEST_ITEM.assets[test_pol].href),
        force=True,
    )

    print("Testing a 1 KiB HTTP range from the remote COG URL...")
    response = requests.get(
        test_href,
        headers={"Range": "bytes=0-1023"},
        timeout=60,
    )

    print("Polarization:", test_pol)
    print("HTTP status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Bytes received:", len(response.content))


Skipping range diagnostic: TEST_ITEM is not defined. Enable the smoke test first if needed.


In [40]:
# ============================================================
# SCENE OBSERVATION CHECKPOINTS + NEAREST-VALID COMPILER
# ============================================================


_SCENE_ARCHIVED_IDS_CACHE = None


def _read_scene_transfer_manifest():
    if not SCENE_TRANSFER_MANIFEST_PATH.exists():
        return {"version": 1, "updated_utc": None, "scenes": {}}
    with SCENE_TRANSFER_MANIFEST_PATH.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    manifest.setdefault("version", 1)
    manifest.setdefault("scenes", {})
    return manifest


def _write_scene_transfer_manifest(manifest):
    from datetime import datetime, timezone
    manifest["updated_utc"] = datetime.now(timezone.utc).isoformat()
    tmp = Path(str(SCENE_TRANSFER_MANIFEST_PATH) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, SCENE_TRANSFER_MANIFEST_PATH)
    global _SCENE_ARCHIVED_IDS_CACHE
    _SCENE_ARCHIVED_IDS_CACHE = {
        str(scene_id)
        for scene_id, entry in manifest.get("scenes", {}).items()
        if entry.get("status") == "archived"
    }


def _scene_is_archived(scene_id):
    global _SCENE_ARCHIVED_IDS_CACHE
    if _SCENE_ARCHIVED_IDS_CACHE is None:
        manifest = _read_scene_transfer_manifest()
        _SCENE_ARCHIVED_IDS_CACHE = {
            str(sid) for sid, entry in manifest.get("scenes", {}).items()
            if entry.get("status") == "archived"
        }
    return str(scene_id) in _SCENE_ARCHIVED_IDS_CACHE


def _sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def prepare_scene_transfer_manifest():
    """Write a transfer manifest for existing shards before rsyncing them to the PC."""
    if not SCENE_INDEX_DB_PATH.exists():
        raise FileNotFoundError(f"Scene inventory missing: {SCENE_INDEX_DB_PATH}")
    manifest = _read_scene_transfer_manifest()
    manifest.setdefault("scenes", {})
    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as conn:
        rows = conn.execute("SELECT scene_id, day FROM scenes ORDER BY day, scene_id").fetchall()

    found = 0
    already_archived = 0
    for scene_id, day in rows:
        scene_id = str(scene_id)
        path = _scene_shard_path(scene_id)
        old = manifest["scenes"].get(scene_id, {})
        if path.exists():
            manifest["scenes"][scene_id] = {
                "filename": path.name,
                "day": int(day),
                "size_bytes": int(path.stat().st_size),
                "sha256": _sha256_file(path),
                "status": "pending",
            }
            found += 1
        elif old.get("status") == "archived":
            already_archived += 1
    _write_scene_transfer_manifest(manifest)
    print(f"Manifest written: {SCENE_TRANSFER_MANIFEST_PATH}")
    print(f"Local shards ready to transfer: {found:,}")
    print(f"Previously archived shards retained in manifest: {already_archived:,}")
    print("Next: rsync the shard directory to your PC with --remove-source-files.")
    print("Only after rsync removes successfully transferred source files, run finalize_scene_transfer_manifest().")
    return SCENE_TRANSFER_MANIFEST_PATH


def finalize_scene_transfer_manifest():
    """Mark only shards removed by a successful rsync --remove-source-files as archived."""
    manifest = _read_scene_transfer_manifest()
    marked = 0
    still_local = 0
    for scene_id, entry in manifest.get("scenes", {}).items():
        if entry.get("status") == "pending":
            local_path = SCENE_OBS_DIR / entry["filename"]
            if not local_path.exists():
                entry["status"] = "archived"
                entry["archived_utc"] = __import__("datetime").datetime.now(
                    __import__("datetime").timezone.utc
                ).isoformat()
                marked += 1
            else:
                still_local += 1
    _write_scene_transfer_manifest(manifest)
    print(f"Marked archived (source file no longer on VM): {marked:,}")
    print(f"Still present locally / not transferred: {still_local:,}")
    print(f"Persistent manifest: {SCENE_TRANSFER_MANIFEST_PATH}")
    print("Resume extraction normally; archived scenes will not be downloaded again.")
    return marked, still_local


def _scene_shard_path(scene_id):
    scene_id = str(scene_id)
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scene_id)
    suffix = hashlib.sha1(scene_id.encode("utf-8")).hexdigest()[:10]
    return SCENE_OBS_DIR / f"{safe[:140]}_{suffix}.npz"


DISK_WARNING_GIB = 10.0
DISK_STOP_GIB = 8.0


class DiskSpacePause(RuntimeError):
    """Controlled checkpoint stop when free disk reaches the configured floor."""


def save_scene_observation(result):
    path = _scene_shard_path(result["item_id"])
    if path.exists():
        return path

    # Preserve room for OS recovery and for safe notebook interruption.
    import shutil
    free_bytes = shutil.disk_usage(OUTPUT_DIR).free
    hard_floor = int(DISK_STOP_GIB * 1024**3)
    if free_bytes < hard_floor:
        raise DiskSpacePause(
            f"LOW_DISK_SPACE_PAUSE: only {free_bytes / 1024**3:.2f} GiB free "
            f"(stop floor {DISK_STOP_GIB:.0f} GiB). No new shard was written."
        )

    tmp_path = Path(str(path) + ".tmp")
    with tmp_path.open("wb") as f:
        np.savez_compressed(
            f,
            station_ids=np.asarray(result["station_ids"], dtype="U"),
            day=np.int64(result["day"]),
            data=result["data"].astype(np.float32, copy=False),
        )
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)
    return path


def run_scene_record(record):
    """Idempotent unique-scene job, checkpointed as a compressed NPZ shard."""
    item = pystac.Item.from_dict(record["item"])
    shard = _scene_shard_path(item.id)

    if shard.exists():
        return {"scene_id": item.id, "status": "cached"}

    scene_cluster_ids = (
        None if record.get("global_scope", False)
        else record.get("cluster_ids", [])
    )
    result = process_scene_with_retry(
        item,
        scene_cluster_ids,
    )

    if result is None:
        result = {
            "item_id": item.id,
            "day": item_day(item),
            "station_ids": np.asarray([], dtype="U"),
            "data": np.empty(
                (0, N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE),
                dtype=np.float32,
            ),
        }
        save_scene_observation(result)
        return {"scene_id": item.id, "status": "empty"}

    save_scene_observation(result)
    return {
        "scene_id": item.id,
        "status": "processed",
        "stations": len(result["station_ids"]),
    }


def _probe_planetary_computer():
    """Cheap preflight so a network outage never starts 200k scene jobs."""
    url = f"{PC_SAS_URL}/{S1_COLLECTION}"
    try:
        response = requests.get(
            url,
            timeout=(PC_TOKEN_CONNECT_TIMEOUT_SECONDS, PC_TOKEN_READ_TIMEOUT_SECONDS),
        )
        response.raise_for_status()
        body = response.json()
        if not isinstance(body, dict) or not body.get("token"):
            raise RuntimeError("Planetary Computer returned an unexpected SAS token response.")
        response.close()
    except Exception as exc:
        raise RuntimeError(
            "Planetary Computer preflight failed. No bulk scene reads were started. "
            "Check outbound HTTPS/network access, then rerun this stage; existing shards and "
            f"the SQLite inventory are preserved. Cause: {type(exc).__name__}: {exc}"
        ) from exc


def run_scene_extraction_from_db(db_path=SCENE_INDEX_DB_PATH, workers=SCENE_WORKERS,
                                 failure_streak_limit=SCENE_FAILURE_STREAK_LIMIT):
    """Resume missing shards; preflight connectivity and stop on repeated remote failures."""
    workers = max(1, int(workers))
    failure_streak_limit = max(1, int(failure_streak_limit))
    _probe_planetary_computer()

    conn = sqlite3.connect(db_path, timeout=120)
    total = int(conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
    cursor = conn.execute(
        """
        SELECT s.scene_id, s.item_json, GROUP_CONCAT(sc.cluster_id)
        FROM scenes AS s
        JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
        GROUP BY s.scene_id
        ORDER BY s.day, s.scene_id
        """
    )

    shard_count = 0
    cached_count = 0
    processed_count = 0
    failures = []
    in_flight = {}
    submitted_at = {}
    exhausted = False
    consecutive_failures = 0
    abort_reason = None
    disk_warning_sent = False
    # Do not build a second executor backlog: each in-flight task maps to one worker.
    max_in_flight = workers
    run_started = time.monotonic()

    print("Unique scenes to check:", f"{total:,}")
    print("Scene workers:", workers)
    print("Maximum submitted scene jobs:", max_in_flight)
    print("GDAL HTTP timeout:", GDAL_ENV["GDAL_HTTP_TIMEOUT"], "seconds per HTTP operation")
    print("Token request timeout:", PC_TOKEN_CONNECT_TIMEOUT_SECONDS, "/", PC_TOKEN_READ_TIMEOUT_SECONDS, "seconds (connect/read)")
    print("Fail-fast threshold:", failure_streak_limit, "consecutive scene failures")

    try:
        with ThreadPoolExecutor(max_workers=workers) as executor:
            with tqdm(
                total=total,
                desc="Scene shards completed",
                unit="scene",
                dynamic_ncols=True,
                mininterval=0.5,
            ) as progress:
                while in_flight or not exhausted:
                    while len(in_flight) < max_in_flight and not exhausted and abort_reason is None:
                        import shutil
                        free_bytes = shutil.disk_usage(OUTPUT_DIR).free
                        free_gib = free_bytes / 1024**3
                        if free_gib < DISK_WARNING_GIB and not disk_warning_sent:
                            tqdm.write(
                                f"LOW-DISK WARNING: {free_gib:.2f} GiB free "
                                f"(warning threshold {DISK_WARNING_GIB:.0f} GiB). "
                                "The pipeline will stop before writing below the configured floor."
                            )
                            disk_warning_sent = True
                        if free_gib < DISK_STOP_GIB:
                            abort_reason = (
                                f"LOW_DISK_SPACE_PAUSE: {free_gib:.2f} GiB free; "
                                f"stop floor is {DISK_STOP_GIB:.0f} GiB."
                            )
                            exhausted = True
                            tqdm.write(
                                abort_reason + " Existing SQLite inventory and completed shards are preserved."
                            )
                            break
                        row = cursor.fetchone()
                        if row is None:
                            exhausted = True
                            break

                        scene_id, item_json, cluster_ids_raw = row
                        if _scene_shard_path(scene_id).exists():
                            shard_count += 1
                            cached_count += 1
                            progress.update(1)
                            continue
                        if _scene_is_archived(scene_id):
                            # The complete NPZ shard was transferred off-VM. Do not
                            # spend time or disk space extracting this scene again.
                            shard_count += 1
                            cached_count += 1
                            progress.update(1)
                            continue

                        record = {
                            "item": json.loads(item_json),
                            "cluster_ids": [
                                int(x) for x in str(cluster_ids_raw).split(",") if x
                            ],
                        }
                        future = executor.submit(run_scene_record, record)
                        in_flight[future] = str(scene_id)
                        submitted_at[future] = time.monotonic()

                    if not in_flight:
                        if abort_reason is not None or exhausted:
                            break
                        continue

                    done_futures, _ = wait(
                        list(in_flight), timeout=2.0, return_when=FIRST_COMPLETED
                    )
                    now = time.monotonic()
                    if not done_futures:
                        active = []
                        for future, scene_id in in_flight.items():
                            age = int(now - submitted_at.get(future, now))
                            state = "running" if future.running() else "queued"
                            active.append(f"{scene_id[:18]}:{state}:{age}s")
                        running_n = sum(1 for future in in_flight if future.running())
                        queued_n = sum(1 for future in in_flight if not future.running() and not future.done())
                        progress.set_postfix(
                            done=f"{shard_count:,}/{total:,}",
                            new=f"{processed_count:,}",
                            cached=f"{cached_count:,}",
                            failed=len(failures),
                            running=running_n,
                            queued=queued_n,
                            current=", ".join(active[:2]) or "none",
                            fail_streak=consecutive_failures,
                            elapsed=f"{(now - run_started) / 60:.1f}m",
                            heartbeat="alive",
                            refresh=True,
                        )
                        continue

                    for future in done_futures:
                        scene_id = in_flight.pop(future)
                        submitted_at.pop(future, None)
                        try:
                            result = future.result()
                            # run_scene_record writes the shard before returning.
                            shard_count += 1
                            processed_count += 1
                            consecutive_failures = 0
                            progress.update(1)
                            progress.set_postfix(
                                status=result.get("status", "ok"),
                                last=scene_id[:18],
                                done=f"{shard_count:,}/{total:,}",
                                failed=len(failures),
                                fail_streak=consecutive_failures,
                                refresh=True,
                            )
                        except KeyboardInterrupt:
                            raise
                        except DiskSpacePause as exc:
                            abort_reason = str(exc)
                            exhausted = True
                            tqdm.write(
                                abort_reason + " Completed shards are kept; the pipeline is pausing for manual intervention."
                            )
                            for pending_future in list(in_flight):
                                if pending_future.cancel():
                                    cancelled_id = in_flight.pop(pending_future)
                                    submitted_at.pop(pending_future, None)
                                    tqdm.write(f"Cancelled not-yet-running scene job {cancelled_id}.")
                        except Exception as exc:
                            failures.append(str(scene_id))
                            consecutive_failures += 1
                            tqdm.write(
                                f"Scene {scene_id} failed "
                                f"({consecutive_failures}/{failure_streak_limit} consecutive): "
                                f"{type(exc).__name__}: {exc}"
                            )
                            progress.set_postfix(
                                done=f"{shard_count:,}/{total:,}",
                                failed=len(failures),
                                fail_streak=consecutive_failures,
                                refresh=True,
                            )
                            if consecutive_failures >= failure_streak_limit:
                                abort_reason = (
                                    f"Stopped after {consecutive_failures} consecutive scene failures. "
                                    "This usually indicates a remote service/network problem or a repeated "
                                    "asset-read error, not a scene-count progress issue."
                                )
                                exhausted = True
                                tqdm.write(abort_reason)
                                # Cancel jobs not yet running; running jobs are allowed to exit
                                # under the reduced GDAL/token timeouts.
                                for pending_future in list(in_flight):
                                    if pending_future.cancel():
                                        cancelled_id = in_flight.pop(pending_future)
                                        submitted_at.pop(pending_future, None)
                                        tqdm.write(f"Cancelled not-yet-running scene job {cancelled_id}.")

        if abort_reason is not None:
            if "LOW_DISK_SPACE_PAUSE" in abort_reason:
                raise RuntimeError(
                    f"{abort_reason} Completed/cached shards observed this run: {shard_count:,}/{total:,}. "
                    "This is a checkpointed pause, not an inventory failure. Download/verify any needed "
                    "backups, free space safely, ensure at least 8 GiB is available, then rerun the final "
                    "run-control cell. The inventory and completed compressed NPZ shards are preserved."
                )
            raise RuntimeError(
                f"{abort_reason} Completed/cached shards: {shard_count:,}/{total:,}. "
                "The database and successful shards are preserved. Restore connectivity or inspect the "
                "first failure, then rerun this stage to resume missing scenes."
            )
        if failures:
            tolerated_failures = sorted(
                set(failures) & OPTIONAL_MISSING_SCENE_IDS
            )
            unapproved_failures = sorted(
                set(failures) - OPTIONAL_MISSING_SCENE_IDS
            )

            if unapproved_failures:
                raise RuntimeError(
                    f"{len(unapproved_failures)} scene jobs failed. "
                    "Successful shards are preserved; rerun this stage to retry "
                    f"missing scenes. First failed IDs: {unapproved_failures[:20]}"
                )

            if tolerated_failures:
                print(
                    "WARNING: proceeding with explicitly tolerated missing scene(s):",
                    tolerated_failures,
                )

    finally:
        cursor.close()
        conn.close()

    print("Scene shards present:", f"{shard_count:,}/{total:,}")
    print(f"Cached shards reused: {cached_count:,}")
    print(f"New scene jobs completed: {processed_count:,}")
    print(f"Scene extraction elapsed: {(time.monotonic() - run_started) / 60:.1f} minutes")


def _load_scene_observation(scene_id):
    path = _scene_shard_path(scene_id)
    if not path.exists():
        if _scene_is_archived(scene_id):
            raise FileNotFoundError(
                f"Scene {scene_id} is archived on your PC, not on the VM. "
                f"Restore its NPZ file ({path.name}) into {SCENE_OBS_DIR} before finalizing. "
                "The missing shard will not be re-extracted automatically."
            )
        return None

    with np.load(path, allow_pickle=False) as z:
        return (
            z["station_ids"].astype(str),
            int(z["day"]),
            z["data"].astype(np.float32, copy=False),
        )


def compile_cluster_year_from_shards(cluster_id, year, samples_year):
    """
    Per sample, select the nearest scene with a valid complete pair:
      - VV/VH pair, and
      - HH/HV pair, independently.

    Nearest means absolute date difference, either before or after the label
    date, no farther than MAX_NEAREST_DAYS. Exact ties prefer earlier dates.
    """
    cluster_samples = samples_year[
        samples_year["cluster_id"] == cluster_id
    ].copy()

    if cluster_samples.empty:
        return None

    sample_days = cluster_samples["sample_date"].to_numpy(dtype="datetime64[D]").astype(np.int64)
    min_day = int(sample_days.min())
    max_day = int(sample_days.max())

    conn = sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120)
    scene_rows = conn.execute(
        """
        SELECT s.scene_id, s.day
        FROM scenes AS s
        JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
        WHERE sc.cluster_id = ?
          AND s.day >= ?
          AND s.day <= ?
        ORDER BY s.day, s.scene_id
        """,
        (
            int(cluster_id),
            min_day - MAX_NEAREST_DAYS,
            max_day + MAX_NEAREST_DAYS,
        ),
    ).fetchall()
    conn.close()
    # Skip only explicitly approved scenes whose local shard is absent.
    optional_missing = {
        str(scene_id)
        for scene_id, _expected_day in scene_rows
        if str(scene_id) in OPTIONAL_MISSING_SCENE_IDS
        and not _scene_shard_path(scene_id).exists()
    }

    if optional_missing:
        print(
            f"Year {year}, cluster {cluster_id}: excluding approved missing "
            f"scene shard(s): {sorted(optional_missing)}"
        )
        scene_rows = [
            (scene_id, expected_day)
            for scene_id, expected_day in scene_rows
            if str(scene_id) not in optional_missing
        ]

    # If archived shards are absent locally, write a precise restore list for this
    # cluster/year before doing any partial aggregation work.
    missing_archived = []
    missing_unarchived = []
    for scene_id, expected_day in scene_rows:
        if not _scene_shard_path(scene_id).exists():
            if _scene_is_archived(scene_id):
                missing_archived.append(_scene_shard_path(scene_id).name)
            else:
                missing_unarchived.append(str(scene_id))
    if missing_archived:
        restore_list = RESTORE_LIST_DIR / f"restore_shards_year_{int(year)}_cluster_{int(cluster_id)}.txt"
        restore_list.write_text("\\n".join(sorted(set(missing_archived))) + "\\n", encoding="utf-8")
        raise FileNotFoundError(
            f"Finalization for year {year}, cluster {cluster_id} needs "
            f"{len(set(missing_archived)):,} archived scene shards. Restore the filenames listed in "
            f"{restore_list} from your PC archive into {SCENE_OBS_DIR}, then rerun the final run-control cell. "
            "Completed Zarr slices are checkpointed and will be skipped."
        )
    if missing_unarchived:
        raise FileNotFoundError(
            f"Finalization for year {year}, cluster {cluster_id} found "
            f"{len(missing_unarchived):,} missing shards that are not marked archived. "
            "Run extraction/resume first; no shards were silently skipped."
        )

    # Build station -> sorted list of scene observations. Only the current
    # cluster's scene associations are loaded; the global inventory stays in SQLite.
    station_observations = {}
    for scene_id, expected_day in scene_rows:
        loaded = _load_scene_observation(scene_id)
        if loaded is None:
            raise FileNotFoundError(
                f"Missing scene shard {scene_id}; run scene extraction first."
            )

        station_ids, stored_day, data = loaded
        for i, station_id in enumerate(station_ids):
            station_observations.setdefault(station_id, []).append(
                (int(stored_day), str(scene_id), data[i])
            )

    for station_id in station_observations:
        station_observations[station_id].sort(
            key=lambda x: (x[0], x[1])
        )

    n = len(cluster_samples)
    output = np.full(
        (n, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE),
        np.nan,
        dtype=np.float32,
    )
    observation_counts = np.zeros(
        (n, N_REAL_BANDS),
        dtype=np.uint8,
    )
    source_day_offset = np.full(
        (n, 2),
        SOURCE_OFFSET_MISSING,
        dtype=np.int16,
    )

    station_ids_for_samples = cluster_samples["station_id"].astype(str).to_numpy()
    pair_definitions = [
        (0, 1, 0),  # VV/VH, offset column 0
        (2, 3, 1),  # HH/HV, offset column 1
    ]

    for row, (station_id, sample_day) in enumerate(zip(station_ids_for_samples, sample_days)):
        entries = station_observations.get(station_id, [])
        if not entries:
            continue

        # Only candidates within the allowed nearest-date interval can be used.
        nearby = [
            (scene_day, scene_id, data)
            for scene_day, scene_id, data in entries
            if abs(int(scene_day) - int(sample_day)) <= MAX_NEAREST_DAYS
        ]
        if not nearby:
            continue

        for band_a, band_b, offset_index in pair_definitions:
            candidates = []

            for scene_day, scene_id, data in nearby:
                a = data[band_a]
                b = data[band_b]
                common_valid = np.isfinite(a) & np.isfinite(b)
                n_valid = int(common_valid.sum())

                if n_valid < MIN_VALID_PAIR_PIXELS:
                    continue

                delta = int(scene_day) - int(sample_day)
                # Rank by absolute distance, then prefer earlier, then more
                # jointly valid 1-km cells, then stable scene ID ordering.
                rank = (
                    abs(delta),
                    0 if delta <= 0 else 1,
                    -n_valid,
                    str(scene_id),
                )
                candidates.append((rank, scene_day, scene_id, data, n_valid))

            if not candidates:
                continue

            observation_counts[row, band_a] = min(len(candidates), 255)
            observation_counts[row, band_b] = min(len(candidates), 255)

            _, chosen_day, _, chosen_data, _ = min(
                candidates,
                key=lambda x: x[0],
            )

            output[row, band_a] = chosen_data[band_a]
            output[row, band_b] = chosen_data[band_b]
            source_day_offset[row, offset_index] = (
                int(chosen_day) - int(sample_day)
            )

    # Ratio is computed only after the aligned VV and VH observations have
    # been selected from the same scene/date.
    vv = output[:, 0]
    vh = output[:, 1]
    ratio = np.full_like(vv, np.nan)
    valid_ratio = np.isfinite(vv) & np.isfinite(vh) & (vh > 0)
    ratio[valid_ratio] = vv[valid_ratio] / vh[valid_ratio]
    output[:, OUTPUT_BANDS.index("vv_vh_ratio")] = ratio

    return output, observation_counts, source_day_offset


In [41]:
# ============================================================
# YEARLY ZARR
# ============================================================

def create_year_zarr(path, n_samples):
    compressor = BloscCodec(
        cname="zstd",
        clevel=3,
        shuffle=BloscShuffle.bitshuffle,
    )

    root = zarr.open_group(path, mode="a")

    if "sample_done" not in root:
        root.attrs.update(
            {
                "schema_version": SCHEMA_VERSION,
                "production_read_strategy": PRODUCTION_READ_STRATEGY,
                "aggregation_method": (
                    "COG overview average resampling to 13x13; approximate"
                    if PRODUCTION_READ_STRATEGY == "overview"
                    else "native 10 m reads and exact 100x100 pixel means"
                ),
                "collection": S1_COLLECTION,
                "patch_size": PATCH_SIZE,
                "model_resolution_m": MODEL_RESOLUTION_M,
                "source_resolution_m": S1_RESOLUTION_M,
                "max_nearest_days": MAX_NEAREST_DAYS,
                "real_bands": REAL_BANDS,
                "output_bands": OUTPUT_BANDS,
                "temporal_selection": (
                    "nearest valid acquisition by absolute date difference; "
                    "earlier date wins exact ties; VV/VH and HH/HV are selected independently"
                ),
                "source_day_offset_columns": [
                    "vv_vh_offset_days",
                    "hh_hv_offset_days",
                ],
                "source_day_offset_missing_value": int(SOURCE_OFFSET_MISSING),
                "vv_vh_ratio": "selected_scene_VV / selected_scene_VH",
                "storage": (
                    "native RTC COG range reads + local 10m-to-1km aggregation "
                    "+ nearest-valid scene selection"
                ),
            }
        )

        root.create_array(
            "sample_done",
            shape=(n_samples,),
            chunks=(min(4096, max(1, n_samples)),),
            dtype="bool",
            fill_value=False,
        )

        for band in OUTPUT_BANDS:
            root.create_array(
                band,
                shape=(n_samples, PATCH_SIZE, PATCH_SIZE),
                chunks=(256, PATCH_SIZE, PATCH_SIZE),
                dtype="float32",
                fill_value=np.nan,
                compressors=compressor,
            )

        root.create_array(
            "obs_count",
            shape=(n_samples, N_REAL_BANDS),
            chunks=(1024, N_REAL_BANDS),
            dtype="uint8",
            fill_value=0,
            compressors=None,
        )

        root.create_array(
            "source_day_offset",
            shape=(n_samples, 2),
            chunks=(1024, 2),
            dtype="int16",
            fill_value=SOURCE_OFFSET_MISSING,
            compressors=None,
        )

    else:
        existing_schema = root.attrs.get("schema_version")
        if existing_schema != SCHEMA_VERSION:
            raise RuntimeError(
                f"Existing Zarr at {path} has schema_version={existing_schema}; "
                f"expected {SCHEMA_VERSION}. Use the versioned nearest_valid_v5 directory."
            )

        if root["sample_done"].shape[0] != n_samples:
            raise RuntimeError("Existing Zarr has a different number of samples.")

        expected_patch_shape = (n_samples, PATCH_SIZE, PATCH_SIZE)
        for band in OUTPUT_BANDS:
            if band not in root or tuple(root[band].shape) != expected_patch_shape:
                raise RuntimeError(
                    f"Existing Zarr band {band!r} is missing or has an incompatible shape."
                )

        if "obs_count" not in root or tuple(root["obs_count"].shape) != (n_samples, N_REAL_BANDS):
            raise RuntimeError("Existing Zarr obs_count has an incompatible shape.")

        if "source_day_offset" not in root or tuple(root["source_day_offset"].shape) != (n_samples, 2):
            raise RuntimeError("Existing Zarr source_day_offset has an incompatible shape.")

    # This attribute is updated on both new and resumed stores to make the current
    # extraction scope explicit; it does not invalidate existing completed samples.
    root.attrs["remote_read_polarizations"] = list(READ_POLARIZATIONS)
    root.attrs["hh_hv_remote_reads_disabled"] = True
    return root


In [42]:
# ============================================================
# YEARLY FINALIZATION
# ============================================================

def prepare_year_metadata(year):
    result = (
        samples_c[samples_c["year"] == year]
        .copy()
        .sort_values(
            ["cluster_id", "station_id", "sample_date"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    result["row_index"] = np.arange(
        len(result),
        dtype=np.int64,
    )
    return result


def run_year(year):
    samples_year = prepare_year_metadata(year)

    metadata_path = FINAL_OUTPUT_DIR / f"samples_nearest_valid_{year}.parquet"
    samples_year.to_parquet(metadata_path, index=False)

    zarr_path = FINAL_OUTPUT_DIR / f"s1_nearest_valid_{year}.zarr"
    root = create_year_zarr(zarr_path, len(samples_year))

    done = np.asarray(root["sample_done"][:], dtype=bool)

    cluster_ranges = (
        samples_year.groupby("cluster_id")["row_index"]
        .agg(start="min", end="max")
        .reset_index()
    )

    import shutil
    warned_low_space = False
    hard_floor_bytes = 8 * 1024**3
    warning_floor_bytes = 10 * 1024**3

    for cluster in tqdm(
        cluster_ranges.itertuples(index=False),
        total=len(cluster_ranges),
        desc=f"Nearest-valid aggregation {year}",
    ):
        cluster_id = int(cluster.cluster_id)
        row_start = int(cluster.start)
        row_end = int(cluster.end)
        row_slice = slice(row_start, row_end + 1)

        if done[row_slice].all():
            continue

        free_bytes = shutil.disk_usage(OUTPUT_DIR).free
        if free_bytes < hard_floor_bytes:
            raise RuntimeError(
                f"Stopping aggregation before cluster {cluster_id} for {year}: "
                f"only {free_bytes / 1024**3:.2f} GiB free; minimum is 8 GiB. "
                "This is a checkpointed pause. Completed sample slices remain checkpointed; free space "
                "safely and rerun the final run-control cell to resume."
            )
        if free_bytes < warning_floor_bytes and not warned_low_space:
            print(
                f"WARNING: disk space is low ({free_bytes / 1024**3:.2f} GiB free). "
                "Aggregation will stop before any cluster write if free space falls below 8 GiB."
            )
            warned_low_space = True

        result = compile_cluster_year_from_shards(
            cluster_id,
            year,
            samples_year,
        )

        if result is None:
            data = np.full(
                (row_end - row_start + 1, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE),
                np.nan,
                dtype=np.float32,
            )
            counts = np.zeros(
                (row_end - row_start + 1, N_REAL_BANDS),
                dtype=np.uint8,
            )
            offsets = np.full(
                (row_end - row_start + 1, 2),
                SOURCE_OFFSET_MISSING,
                dtype=np.int16,
            )
        else:
            data, counts, offsets = result

        # The cluster rows are contiguous because samples_year is sorted by cluster.
        if len(data) != (row_end - row_start + 1):
            raise RuntimeError(
                f"Cluster {cluster_id} has {len(data)} rows, but its Zarr slice expects "
                f"{row_end - row_start + 1}."
            )

        for index, band in enumerate(OUTPUT_BANDS):
            root[band][row_slice] = data[:, index]

        root["obs_count"][row_slice] = counts
        root["source_day_offset"][row_slice] = offsets

        # Set completion only after all feature arrays have been written.
        root["sample_done"][row_slice] = True
        done[row_slice] = True

    root.attrs["complete"] = bool(
        np.asarray(root["sample_done"][:], dtype=bool).all()
    )

    return metadata_path, zarr_path


## Resume workflow and checkpoints

This notebook is configured to reuse the existing `scene_inventory_nearest_v1.sqlite`, the saved `stations_with_clusters.parquet` mapping, and the `scene_observations_nearest_overview_v1` scene shards. The saved cluster IDs are loaded rather than recomputed because the SQLite `scene_clusters` table depends on those exact IDs.

Open the notebook from the project root (`~/sentinel1-rtc`) and run the setup/definition cells in order with the final run-control flags left `False`. Do not use **Run All** after enabling the resume flag. Once setup and the inventory validation look correct, set `RUN_RESUME_EXISTING_PIPELINE = True` in the final cell and run that cell only.

Resume mode does not rebuild the inventory. It verifies all cluster searches are complete, checks for inconsistent cluster-search records, resumes only missing scene shards, and then checkpoints nearest-valid yearly output. Existing completed outputs are preserved.

The production reader remains in the previously selected `overview` mode for throughput. This is approximate relative to exact native 10 m → 1 km pixel means; it is intentionally not switched by the optional benchmark cell.


Each scene is checkpointed as an atomic `np.savez_compressed` NPZ shard; yearly arrays use Zstandard compression. The run stops at less than 8 GiB free, preserving checkpoints. For repeated disk-pressure offloads, use the transfer workflow at the end of this notebook: transfer completed NPZ shards to your PC with rsync --remove-source-files, mark successfully removed shards in the persistent transfer manifest, and resume extraction without recomputing those scenes. Final aggregation still needs the shard contents, so restore only the shard files requested by its generated restore list before finalizing each year.


Scene concurrency is explicitly set to 8 workers as requested. This overrides RAM-based auto-detection; polarization reads remain serial within each scene to limit additional memory pressure. On the 7.8 GiB VM, monitor memory/swap and failures closely; if the machine starts swapping heavily or workers fail repeatedly, reduce `SCENE_WORKERS`.


## Optional benchmark: exact 10 m → 1 km aggregation vs COG overview reads

The global resume does not require rerunning this benchmark. The chosen production strategy is `overview`, because throughput is the priority for the remaining scene extraction. Overview-resampled values are approximate relative to the exact native-resolution 100 × 100 pixel means; earlier benchmark results showed that trade-off.

This optional benchmark reads a small sample of scene patches and reports differences between direct COG-overview resampling and exact aggregation. It is disabled by default. Do not enable it during a resume unless you specifically want to revisit the accuracy/speed trade-off. The global resume flag remains separate and disabled by default.


In [43]:
# ============================================================
# COG OVERVIEW BENCHMARK -- EXACT REFERENCE VS DIRECT OVERVIEW READ
# ============================================================

from rasterio.enums import Resampling

RUN_COG_OVERVIEW_BENCHMARK = False
COG_BENCHMARK_SCENE_COUNT = 3
COG_BENCHMARK_STATIONS_PER_SCENE = 2
COG_BENCHMARK_CANDIDATES_PER_TIME_REGION = 36
COG_BENCHMARK_POLARIZATION_PAIRS = (("vv", "vh"),)  # VV/VH-only; never benchmark HH/HV


def _benchmark_pair_for_item(item):
    available = set(available_polarizations(item))
    for pair in COG_BENCHMARK_POLARIZATION_PAIRS:
        if set(pair).issubset(available):
            return tuple(pair)
    return None


def _benchmark_normalize_values(values, nodata):
    values = np.asarray(values, dtype=np.float32).copy()
    if nodata is not None:
        try:
            if not np.isnan(float(nodata)):
                values[values == np.float32(nodata)] = np.nan
        except (TypeError, ValueError):
            pass
    values[~np.isfinite(values)] = np.nan
    return values


def _benchmark_candidate_scene_rows(db_path, candidates_per_region=36):
    """Fetch small, evenly spaced slices from the day-sorted SQLite index."""
    if not Path(db_path).exists():
        print(f"Benchmark skipped: scene inventory database not found: {db_path}")
        return []

    conn = sqlite3.connect(db_path, timeout=60)
    try:
        count, min_day, max_day = conn.execute(
            "SELECT COUNT(*), MIN(day), MAX(day) FROM scenes"
        ).fetchone()
        count = int(count or 0)
        if count == 0:
            print("Benchmark skipped: scene inventory database is empty.")
            return []

        # Keep the time regions separate so the scene chooser must try one scene
        # from early, middle, and late acquisition periods before filling extras.
        quantiles = np.linspace(0.06, 0.94, int(COG_BENCHMARK_SCENE_COUNT))
        offsets = [int(round(float(q) * (count - 1))) for q in quantiles]
        per_region = max(8, int(candidates_per_region))
        regions = []
        seen = set()
        for offset in offsets:
            start = max(0, offset - per_region // 2)
            batch = conn.execute(
                "SELECT scene_id, day, item_json FROM scenes "
                "ORDER BY day, scene_id LIMIT ? OFFSET ?",
                (per_region, start),
            ).fetchall()
            unique_batch = []
            for row in batch:
                if row[0] not in seen:
                    seen.add(row[0])
                    unique_batch.append(row)
            regions.append(unique_batch)

        date_start = (pd.Timestamp("1970-01-01") + pd.Timedelta(days=int(min_day))).date()
        date_end = (pd.Timestamp("1970-01-01") + pd.Timedelta(days=int(max_day))).date()
        print(
            f"Benchmark candidate pool: {sum(map(len, regions))} rows sampled from "
            f"{count:,} scenes ({date_start} to {date_end}); "
            f"regions={[len(x) for x in regions]}"
        )
        return regions
    finally:
        conn.close()

def _choose_cog_benchmark_scenes(
    db_path=SCENE_INDEX_DB_PATH,
    scene_count=COG_BENCHMARK_SCENE_COUNT,
    stations_per_scene=COG_BENCHMARK_STATIONS_PER_SCENE,
    candidates_per_region=COG_BENCHMARK_CANDIDATES_PER_TIME_REGION,
):
    """Choose scenes spanning the indexed timeline, with overviews and eligible patches."""
    candidate_regions = _benchmark_candidate_scene_rows(
        db_path, candidates_per_region=candidates_per_region
    )
    if not candidate_regions:
        return []

    selected = []
    seen_ids = set()

    def evaluate_candidate(row):
        scene_id, scene_day, item_json = row
        if scene_id in seen_ids:
            return None
        seen_ids.add(scene_id)
        try:
            item = pystac.Item.from_dict(json.loads(item_json))
            pair = _benchmark_pair_for_item(item)
            if pair is None:
                return None

            station_candidates = _global_scene_station_candidates(item)
            needed = stations_needed_for_scene(item, station_candidates, SAMPLE_DAYS)
            if needed.empty:
                return None

            # The metadata open is intentionally outside timed measurements. It
            # verifies real internal overviews and finds fully covered 1300 px patches.
            href = rtc_token.sign(_clean_href(item.assets[pair[0]].href), force=False)
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(href) as src:
                    overviews = list(src.overviews(1))
                    if not overviews:
                        return None
                    pixels, pixel_valid = station_patch_windows(
                        src.transform, src.crs.to_string(), src.width, src.height, needed
                    )
                    fully_covered = (
                        pixel_valid
                        & (pixels[:, 0] >= 0)
                        & (pixels[:, 1] >= 0)
                        & (pixels[:, 2] <= src.width)
                        & (pixels[:, 3] <= src.height)
                        & ((pixels[:, 2] - pixels[:, 0]) == SOURCE_PATCH_PX)
                        & ((pixels[:, 3] - pixels[:, 1]) == SOURCE_PATCH_PX)
                    )
                    eligible = np.flatnonzero(fully_covered)
                    if len(eligible) == 0:
                        return None
                    take = min(int(stations_per_scene), len(eligible))
                    positions = np.linspace(0, len(eligible) - 1, take).round().astype(int)
                    chosen_positions = eligible[positions]
                    station_rows = []
                    for pos in chosen_positions:
                        row_data = needed.iloc[int(pos)]
                        c0, r0, c1, r1 = map(int, pixels[int(pos)])
                        station_rows.append({
                            "station_id": str(row_data["station_id"]),
                            "window": (c0, r0, c1, r1),
                        })

            result = {
                "item": item,
                "scene_id": str(item.id),
                "day": int(scene_day),
                "polarizations": pair,
                "overview_factors": overviews,
                "stations": station_rows,
            }
            print(
                f"Selected {item.id}: day={int(scene_day)}, pair={pair}, "
                f"overviews={overviews}, patches={len(station_rows)}"
            )
            return result
        except (
            OSError, ValueError, KeyError, TypeError,
            rasterio.errors.RasterioIOError, requests.RequestException,
        ) as exc:
            print(f"Skipping benchmark candidate {scene_id}: {type(exc).__name__}: {str(exc)[:140]}")
            return None

    # First aim for one representative scene from each temporal region.
    for region in candidate_regions:
        for row in region:
            candidate = evaluate_candidate(row)
            if candidate is not None:
                selected.append(candidate)
                break
        if len(selected) >= int(scene_count):
            break

    # If a region had no eligible scenes, fill from remaining sampled candidates.
    if len(selected) < int(scene_count):
        for region in candidate_regions:
            for row in region:
                candidate = evaluate_candidate(row)
                if candidate is not None:
                    selected.append(candidate)
                    if len(selected) >= int(scene_count):
                        break
            if len(selected) >= int(scene_count):
                break

    if len(selected) < int(scene_count):
        print(
            f"Warning: found {len(selected)} eligible scenes, requested {scene_count}. "
            "The benchmark will use those that qualified."
        )
    return selected

def _benchmark_read_patch(href, asset_key, window_tuple, strategy):
    """Time a fresh COG open plus one patch read. SAS-token time is excluded."""
    c0, r0, c1, r1 = map(int, window_tuple)
    window = Window(c0, r0, c1 - c0, r1 - r0)
    benchmark_env = dict(GDAL_ENV)
    # Disable per-handle VSI caching for both arms so one strategy doesn't
    # inherit its counterpart's local file-handle block cache.
    benchmark_env["VSI_CACHE"] = "FALSE"
    started = time.perf_counter()
    with rasterio.Env(**benchmark_env):
        with rasterio.open(href) as src:
            if strategy == "exact":
                native = read_block(src, window)
                result = pool_10m_to_1km(native)
            elif strategy == "overview":
                result = src.read(
                    1,
                    window=window,
                    out_shape=(PATCH_SIZE, PATCH_SIZE),
                    out_dtype="float32",
                    resampling=Resampling.average,
                )
                result = _benchmark_normalize_values(result, src.nodata)
            else:
                raise ValueError(f"Unknown benchmark strategy: {strategy}")
    return result, float(time.perf_counter() - started)


def _benchmark_patch_metrics(exact, overview):
    exact = np.asarray(exact, dtype=np.float32)
    overview = np.asarray(overview, dtype=np.float32)
    exact_valid = np.isfinite(exact)
    overview_valid = np.isfinite(overview)
    joint = exact_valid & overview_valid
    mask_agreement = float(np.mean(exact_valid == overview_valid))
    out = {
        "valid_exact": int(exact_valid.sum()),
        "valid_overview": int(overview_valid.sum()),
        "valid_joint": int(joint.sum()),
        "valid_mask_agreement_pct": 100.0 * mask_agreement,
        "mae": np.nan,
        "rmse": np.nan,
        "bias_overview_minus_exact": np.nan,
        "p95_abs_error": np.nan,
        "max_abs_error": np.nan,
        "mae_pct_of_mean_abs_exact": np.nan,
    }
    if joint.any():
        difference = overview[joint].astype(np.float64) - exact[joint].astype(np.float64)
        absolute = np.abs(difference)
        mae = float(np.mean(absolute))
        mean_abs_exact = float(np.mean(np.abs(exact[joint].astype(np.float64))))
        out.update({
            "mae": mae,
            "rmse": float(np.sqrt(np.mean(difference ** 2))),
            "bias_overview_minus_exact": float(np.mean(difference)),
            "p95_abs_error": float(np.percentile(absolute, 95)),
            "max_abs_error": float(np.max(absolute)),
            "mae_pct_of_mean_abs_exact": float(100.0 * mae / mean_abs_exact) if mean_abs_exact > 0 else np.nan,
        })
    return out


def run_cog_overview_benchmark(
    db_path=SCENE_INDEX_DB_PATH,
    scene_count=COG_BENCHMARK_SCENE_COUNT,
    stations_per_scene=COG_BENCHMARK_STATIONS_PER_SCENE,
):
    """Compare exact patch means with COG overview-based 13x13 reads on real scenes."""
    if not Path(db_path).exists():
        print(f"Benchmark skipped: scene inventory database not found: {db_path}")
        return pd.DataFrame(), pd.DataFrame()
    with sqlite3.connect(db_path, timeout=60) as _conn:
        _count = int(_conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
    if _count == 0:
        print("Benchmark skipped: scene inventory database contains no scenes.")
        return pd.DataFrame(), pd.DataFrame()
    chosen = _choose_cog_benchmark_scenes(
        db_path=db_path,
        scene_count=scene_count,
        stations_per_scene=stations_per_scene,
    )
    if not chosen:
        raise RuntimeError(
            "No eligible COG scenes with overviews and fully covered station patches were found. "
            f"PRODUCTION_READ_STRATEGY remains {PRODUCTION_READ_STRATEGY!r}."
        )

    rows = []
    task_index = 0
    for scene in chosen:
        item = scene["item"]
        for pol in scene["polarizations"]:
            base_href = _clean_href(item.assets[pol].href)
            # Make method-specific signed URLs before timing, reducing the chance
            # that one method reuses the other's GDAL URL/block-cache entries.
            exact_token = rtc_token.token(force=True)
            overview_token = rtc_token.token(force=True)
            exact_href = f"{base_href}?{exact_token}"
            overview_href = f"{base_href}?{overview_token}"

            for station in scene["stations"]:
                # Alternate order to counteract systematic warm/cold ordering.
                if task_index % 2 == 0:
                    order = (("exact", exact_href), ("overview", overview_href))
                else:
                    order = (("overview", overview_href), ("exact", exact_href))
                task_index += 1
                values = {}
                durations = {}
                for strategy, href in order:
                    array, seconds = _benchmark_read_patch(
                        href, pol, station["window"], strategy
                    )
                    values[strategy] = array
                    durations[strategy] = seconds

                metrics = _benchmark_patch_metrics(values["exact"], values["overview"])
                exact_seconds = durations["exact"]
                overview_seconds = durations["overview"]
                rows.append({
                    "scene_id": scene["scene_id"],
                    "scene_day": int(scene["day"]),
                    "station_id": station["station_id"],
                    "polarization": pol,
                    "overview_factors": ",".join(map(str, scene["overview_factors"])),
                    "exact_read_seconds": exact_seconds,
                    "overview_read_seconds": overview_seconds,
                    "speedup_exact_overview": exact_seconds / overview_seconds if overview_seconds > 0 else np.nan,
                    **metrics,
                })
                print(
                    f"{scene['scene_id']} | {station['station_id']} | {pol}: "
                    f"exact={exact_seconds:.2f}s, overview={overview_seconds:.2f}s, "
                    f"speedup={exact_seconds / overview_seconds if overview_seconds > 0 else np.nan:.2f}x, "
                    f"MAE={metrics['mae']:.6g}, RMSE={metrics['rmse']:.6g}, "
                    f"mask agreement={metrics['valid_mask_agreement_pct']:.1f}%"
                )

    results = pd.DataFrame(rows)
    if results.empty:
        raise RuntimeError("Benchmark produced no measurements; PRODUCTION_READ_STRATEGY is unchanged.")

    summary = results.groupby("polarization", dropna=False).agg(
        patch_tests=("scene_id", "count"),
        median_exact_seconds=("exact_read_seconds", "median"),
        median_overview_seconds=("overview_read_seconds", "median"),
        median_speedup=("speedup_exact_overview", "median"),
        median_mae=("mae", "median"),
        median_rmse=("rmse", "median"),
        worst_patch_p95_abs_error=("p95_abs_error", "max"),
        min_mask_agreement_pct=("valid_mask_agreement_pct", "min"),
        median_relative_mae_pct=("mae_pct_of_mean_abs_exact", "median"),
    ).reset_index()

    timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
    result_path = OUTPUT_DIR / f"cog_overview_benchmark_{timestamp}.csv"
    summary_path = OUTPUT_DIR / f"cog_overview_benchmark_summary_{timestamp}.csv"
    results.to_csv(result_path, index=False)
    summary.to_csv(summary_path, index=False)

    print("\n=== COG overview benchmark summary ===")
    print(summary.to_string(index=False))
    print("\nPer-patch results:", result_path)
    print("Summary:", summary_path)
    print(
        "\nThe benchmark does not change PRODUCTION_READ_STRATEGY. "
        f"Current production strategy: {PRODUCTION_READ_STRATEGY!r}. "
        "Overview mode is approximate; set the configuration to 'exact' if the "
        "measured numerical differences are not acceptable for the application."
    )
    return results, summary


if RUN_COG_OVERVIEW_BENCHMARK:
    cog_benchmark_results, cog_benchmark_summary = run_cog_overview_benchmark()
else:
    print("COG overview benchmark is disabled by default. Set RUN_COG_OVERVIEW_BENCHMARK=True to rerun it.")


COG overview benchmark is disabled by default. Set RUN_COG_OVERVIEW_BENCHMARK=True to rerun it.


In [44]:
# ============================================================
# GLOBAL RUN CONTROL: EXPLICIT RESUME, NO AUTOMATIC INVENTORY REBUILD
# ============================================================

# Leave all flags False while opening the notebook and running setup cells.
# For a resume, set RUN_RESUME_EXISTING_PIPELINE=True and execute THIS CELL ONLY.
# This mode validates the existing SQLite checkpoints, resumes only missing scene
# shards, and then resumes/creates the nearest-valid yearly Zarr outputs. It does
# NOT rebuild the 819-cluster inventory.
#
# RUN_BUILD_INVENTORY_ONLY is for a deliberate metadata-only maintenance run.
# RUN_FULL_GLOBAL_PIPELINE is for a deliberate inventory + extraction run from the
# beginning. Do not enable both modes together.
RUN_RESUME_EXISTING_PIPELINE = True
RUN_BUILD_INVENTORY_ONLY = False
RUN_FULL_GLOBAL_PIPELINE = False

_active_modes = sum((
    bool(RUN_RESUME_EXISTING_PIPELINE),
    bool(RUN_BUILD_INVENTORY_ONLY),
    bool(RUN_FULL_GLOBAL_PIPELINE),
))
if _active_modes > 1:
    raise ValueError("Choose only one global run mode.")

def _print_inventory_summary(require_complete=True):
    if not SCENE_INDEX_DB_PATH.exists():
        raise FileNotFoundError(
            f"Existing scene inventory not found: {SCENE_INDEX_DB_PATH}. "
            "Copy scene_inventory_nearest_v1.sqlite before resuming."
        )
    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as conn:
        n_scenes = int(conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
        n_associations = int(conn.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0])
        scenes_without_cluster_links = int(conn.execute(
            """
            SELECT COUNT(*) FROM scenes AS s
            LEFT JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
            WHERE sc.scene_id IS NULL
            """
        ).fetchone()[0])
        completed = {int(r[0]) for r in conn.execute(
            "SELECT cluster_id FROM cluster_searches WHERE completed=1"
        )}
        expected = {int(x) for x in clusters["cluster_id"].unique()}
        recorded = {int(r[0]) for r in conn.execute(
            "SELECT DISTINCT cluster_id FROM cluster_searches"
        )}
        bad_empty = conn.execute(
            """
            SELECT cs.cluster_id, cs.result_count, COUNT(sc.scene_id) AS association_count
            FROM cluster_searches AS cs
            LEFT JOIN scene_clusters AS sc ON sc.cluster_id = cs.cluster_id
            GROUP BY cs.cluster_id, cs.result_count
            HAVING COUNT(sc.scene_id) = 0 AND COALESCE(cs.result_count, 0) > 0
            ORDER BY cs.cluster_id
            """
        ).fetchall()
        unknown_assoc = {int(r[0]) for r in conn.execute(
            "SELECT DISTINCT cluster_id FROM scene_clusters"
        )} - expected

    if recorded != expected:
        raise RuntimeError(
            "SQLite cluster IDs differ from the saved station mapping: "
            f"expected {len(expected)} IDs, found {len(recorded)}."
        )
    incomplete = expected - completed
    if require_complete and incomplete:
        raise RuntimeError(
            f"Inventory has {len(incomplete)} incomplete cluster searches; "
            "resume paused rather than silently rebuild or skip them."
        )
    if bad_empty:
        raise RuntimeError(
            "Some completed clusters report results but have no scene-cluster links; "
            f"first problematic records: {bad_empty[:5]}. Inventory needs investigation."
        )
    if scenes_without_cluster_links:
        raise RuntimeError(
            f"SQLite contains {scenes_without_cluster_links:,} scenes with no scene-cluster links; "
            "resume paused rather than silently skip them."
        )
    if unknown_assoc:
        raise RuntimeError(
            f"Inventory references cluster IDs absent from the saved map: {sorted(unknown_assoc)[:10]}"
        )

    print("Inventory DB:", SCENE_INDEX_DB_PATH)
    print("Unique scenes:", f"{n_scenes:,}")
    print("Scene-cluster associations:", f"{n_associations:,}")
    print("Scenes lacking cluster links:", f"{scenes_without_cluster_links:,}")
    print("Completed cluster searches:", f"{len(completed):,}/{len(expected):,}")
    if require_complete:
        print("Inventory checkpoint validation: PASS")
    return n_scenes, n_associations

if RUN_BUILD_INVENTORY_ONLY:
    print("Building/resuming metadata inventory only; no raster scene extraction will run.")
    build_global_scene_index()
elif RUN_RESUME_EXISTING_PIPELINE:
    print("RESUMING EXISTING INVENTORY + OVERVIEW SHARDS")
    print("Output namespace:", OUTPUT_DIR)
    print("Read strategy:", PRODUCTION_READ_STRATEGY, "| nearest tolerance:", MAX_NEAREST_DAYS, "days")
    print("Workers:", SCENE_WORKERS, f"(detected RAM {TOTAL_RAM_GIB:.1f} GiB)", "| remote polarizations:", READ_POLARIZATIONS)
    _print_inventory_summary(require_complete=True)

    print("\n" + "=" * 70)
    print("1/2 RESUME MISSING UNIQUE-SCENE SHARDS")
    print("=" * 70)
    run_scene_extraction_from_db(
        db_path=SCENE_INDEX_DB_PATH,
        workers=SCENE_WORKERS,
    )

    print("\n" + "=" * 70)
    print("2/2 RESUME NEAREST-VALID YEARLY FINALIZATION")
    print("=" * 70)
    years = sorted(samples_c["year"].dropna().astype(int).unique())
    print("Years:", years[0], "->", years[-1])
    print("Total sample rows:", f"{len(samples_c):,}")
    for year in years:
        print(f"\nFinalizing {year}...")
        metadata_path, zarr_path = run_year(int(year))
        print("Metadata:", metadata_path)
        print("Zarr:", zarr_path)
    print("Nearest-valid yearly finalization reached end of configured years.")
elif RUN_FULL_GLOBAL_PIPELINE:
    years = sorted(samples_c["year"].dropna().astype(int).unique())
    print("FULL GLOBAL PIPELINE MODE: this will build/resume the inventory before scene extraction.")
    print("Years:", years[0], "->", years[-1])
    print("Total samples:", f"{len(samples_c):,}")
    print("Total stations:", f"{stations_cluster.station_id.nunique():,}")
    print("Total clusters:", clusters["cluster_id"].nunique())
    print("Building/resuming inventory...")
    build_global_scene_index()
    _print_inventory_summary(require_complete=True)
    run_scene_extraction_from_db(db_path=SCENE_INDEX_DB_PATH, workers=SCENE_WORKERS)
    for year in years:
        print(f"\nFinalizing {year}...")
        metadata_path, zarr_path = run_year(int(year))
        print("Metadata:", metadata_path)
        print("Zarr:", zarr_path)
    print("Full global pipeline reached end of configured years.")
else:
    print("No global work started. Setup/validation cells can be run safely.")
    print("To resume later, enable RUN_RESUME_EXISTING_PIPELINE in this cell and execute this cell only.")
    print("Output namespace:", OUTPUT_DIR)
    print("Scene shards:", SCENE_OBS_DIR)
    print("Inventory DB:", SCENE_INDEX_DB_PATH)
    print("Final outputs:", FINAL_OUTPUT_DIR)


RESUMING EXISTING INVENTORY + OVERVIEW SHARDS
Output namespace: Data/Global/pipeline_stage1/sentinel1
Read strategy: overview | nearest tolerance: 18 days
Workers: 8 (detected RAM 7.8 GiB) | remote polarizations: ('vv', 'vh')


Inventory DB: Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
Unique scenes: 204,033
Scene-cluster associations: 528,280
Scenes lacking cluster links: 0
Completed cluster searches: 819/819
Inventory checkpoint validation: PASS

1/2 RESUME MISSING UNIQUE-SCENE SHARDS


KeyboardInterrupt: 

## Disk-pressure transfer workflow — Windows SSH, resumable extraction

The scene shards are checkpointed individually. When extraction reaches the 8 GiB free-space floor, it pauses without rebuilding the SQLite inventory.

### When the low-disk pause occurs

1. Wait for the `LOW_DISK_SPACE_PAUSE` message and the end of the run-control cell. Do not restart the kernel or delete any files yet.
2. Run the transfer-preparation cell below once. It records hashes and sizes for the existing local shards in `scene_transfer_manifest_nearest_v1.json`.
3. From **Windows Command Prompt**, stream a batch of 5,000 scene shards directly from the VM into a compressed archive on `D:\SentinelRTC`. This uses the SSH alias configured on Windows, not WSL. No temporary archive is created on the VM.
4. Verify the downloaded archive with `tar -tzf` before deleting any VM shards.
5. Only after verification succeeds, delete the same 5,000 source shards using the Windows SSH connection. Then return to the notebook and run `finalize_scene_transfer_manifest()`.
6. Repeat with the next batch number until enough VM disk space is free.
7. Rerun the final run-control cell with only `RUN_RESUME_EXISTING_PIPELINE = True`.

### Important

- Keep `scene_inventory_nearest_v1.sqlite`, the transfer manifest, the station/cluster mapping, and completed yearly outputs on the VM.
- Keep the downloaded archives on Windows and retain a second copy of the archive data.
- Never delete VM shards before their corresponding Windows archive passes the integrity check.
- Archived shards are skipped during extraction. Final aggregation still needs their contents, so restore only the shards requested by the notebook's restore list when it reaches that stage.


In [ ]:
# ============================================================
# PREPARE A SAFE WINDOWS-SSH SHARD OFFLOAD
# ============================================================
# Run this ONCE after a low-disk pause, before transferring batches.
# It hashes existing NPZ shards and marks them pending in the manifest.
# Do not rerun until needed for a new offload cycle.

RUN_PREPARE_SCENE_TRANSFER_MANIFEST = False

if RUN_PREPARE_SCENE_TRANSFER_MANIFEST:
    prepare_scene_transfer_manifest()
    print()
    print("MANIFEST PREPARED.")
    print("Next: use Windows Command Prompt to stream a 5,000-shard batch.")
    print("Verify each downloaded archive before deleting its source shards.")
    print("After each verified batch is deleted from the VM, run:")
    print("    finalize_scene_transfer_manifest()")
else:
    print("Transfer preparation is disabled.")
    print("Set RUN_PREPARE_SCENE_TRANSFER_MANIFEST = True and run this cell")
    print("only after the extraction stage has paused for low disk space.")
